In [1]:
!pip install opencv-python deepface SpeechRecognition sounddevice PyAudio
!pip install tf-keras
!pip install opencv-python deepface SpeechRecognition sounddevice numpy ipywidgets
!pip install transformers torch sentencepiece 
!pip install google-generativeai 
!pip install pyttsx3 
!pip install pypiwin32
!pip install huggingface_hub[hf_xet] 
!pip install hf_xet

In [2]:
# ========================
# FINAL INTEGRATED PROTOTYPE CODE (Enhanced with Live Voice SER/Spectrogram)
# ========================
# Run this cell first for installs!
!pip install ipywidgets matplotlib numpy librosa tensorflow scikit-learn transformers sounddevice SpeechRecognition pyttsx3 google-generativeai pydub 
# pydub helps handle audio data

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 1.3 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/8.1 MB 1.3 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/8.1 MB 1.3 MB/s eta 0:00:06
   ------ --------------------------------- 1.3/8.1 MB 1.3 MB/s eta 0:00:06
   ------- -------------------------------- 1.6/8.1 MB 1.3 MB/s eta 0:00:06
   --------- --

In [8]:
# ========================
# FINAL INTEGRATED PROTOTYPE CODE (Enhanced with Live Voice/Face SER/Spectrogram in VIDEO mode)
# ========================
# Run this cell first for installs!
# %pip install ipywidgets matplotlib numpy librosa tensorflow scikit-learn transformers sounddevice SpeechRecognition pyttsx3 google-generativeai pydub deepface opencv-python # Added deepface, opencv

# --- Step 1: Imports & Setup ---
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
import matplotlib.pyplot as plt
import numpy as np
import random
import datetime
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from transformers import pipeline
import sounddevice as sd
import speech_recognition as sr
import pyttsx3
import google.generativeai as genai
import threading
import queue
import time
import os
import logging
from pydub import AudioSegment
import io
import cv2 # Added for video capture
from deepface import DeepFace # Added for facial emotion recognition

# For reproducibility and silencing TF/DeepFace warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
# logging.getLogger("tensorflow").setLevel(logging.ERROR) # Alternate way
# logging.getLogger("deepface").setLevel(logging.WARN)    # Reduce deepface spam
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries loaded.")

# --- Global Config & Resources ---
AUDIO_SAMPLE_RATE = 16000
AUDIO_CHANNELS = 1
AUDIO_CHUNK_DURATION = 0.5
AUDIO_CHUNK_SAMPLES = int(AUDIO_SAMPLE_RATE * AUDIO_CHUNK_DURATION)
STT_PHRASE_TIME_LIMIT = 7
# Video/DeepFace settings
VIDEO_DETECTION_INTERVAL = 5 # Frames between face detection runs
DEEPFACE_DETECTOR_BACKEND = 'opencv' # Fast but less accurate; try 'mtcnn', 'ssd' etc. if needed

# Queues & Events
stt_text_q = queue.Queue()
tts_q = queue.Queue()
audio_chunk_q = queue.Queue(maxsize=20)
video_frame_q = queue.Queue(maxsize=5) # Queue for frames to display widget
stop_event = threading.Event()
listen_event = threading.Event()

# Shared State (with locks)
chat_history_lock = threading.Lock()
chat_history = []
voice_emotion_log_lock = threading.Lock()
voice_emotion_log = [] # List of (timestamp, detected_emotion, confidence)
face_emotion_log_lock = threading.Lock()
face_emotion_log = [] # List of (timestamp, detected_emotion, confidence)

# Global emotion holders (updated by threads)
last_detected_voice_emotion_global = "N/A"
last_detected_face_emotion_global = "N/A"

# Matplotlib Plot variables (for spectrogram)
ser_fig = None
ser_ax = None
spectrogram_output_widget = None # Will be set depending on chat mode

print("Shared resources initialized.")

Libraries loaded.
Shared resources initialized.


In [5]:
# --- Step 2: User Authentication (From main_prototype -UNCHANGED) ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

user_db = {} # Simulated user database

def create_account(username, password):
    logging.info(f"Attempting account creation for: {username}")
    if not username or not password: # Add validation
         logging.warning("Signup failed: Username or password empty.")
         return False, "Username and password cannot be empty."
    if username in user_db:
        logging.warning(f"Signup failed: Username '{username}' already exists.")
        return False, "Username already exists."
    else:
        # Simple password "hashing" for prototype - DO NOT USE IN PRODUCTION
        # In production use bcrypt: import bcrypt; hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
        user_db[username] = {"password": password, "data": {}}
        logging.info(f"Account created successfully for: {username}")
        return True, "Account created successfully. Please Login."

def login(username, password):
    logging.info(f"Attempting login for: {username}")
    if not username or not password:
        logging.warning("Login failed: Username or password empty.")
        return False, "Username and password cannot be empty."
    if username in user_db and user_db[username]["password"] == password: # Compare plain text (prototype only)
        # In production: bcrypt.checkpw(password.encode(), user_db[username]["password_hash"].encode())
        logging.info(f"Login successful for: {username}")
        return True, "Login successful."
    else:
        logging.warning(f"Login failed: Invalid credentials for '{username}'.")
        return False, "Invalid username or password."

# Widgets for login (Ensure these are defined only once globally if needed elsewhere)
login_username = widgets.Text(description="Username:")
login_password = widgets.Password(description="Password:")
login_button = widgets.Button(description="Login", button_style='success') # Added style
signup_button = widgets.Button(description="Sign Up", button_style='info') # Added style
login_output = widgets.Output(layout={'border': '1px solid black', 'margin_top': '10px'}) # Added border/margin
main_area = widgets.Output() # Area to display next steps after login
current_username = None # Store logged in user globally

# --- Define Callbacks WITH Error Handling ---
def on_login_clicked(b):
    global current_username
    print("--- Login Clicked ---") # Print to notebook output for basic check
    with login_output: # Ensure output goes to the widget
        clear_output(wait=True) # Clear previous messages in this widget
        logging.info("Login button event triggered.")
        try:
            user = login_username.value
            pwd = login_password.value
            logging.info(f"Attempting login for user: '{user}'")
            success, msg = login(user, pwd)
            print(msg) # Print result message inside the login_output widget
            if success:
                logging.info("Login success branch entered.")
                current_username = user
                # --- UI Transition ---
                login_box.layout.display = 'none' # Hide the login box
                with main_area: # Use the main_area Output widget for the next step
                    clear_output(wait=True) # Clear anything previously in main_area
                    logging.info("Calling show_intake_form...")
                    show_intake_form(user) # Display the intake form HERE
            else:
                 logging.warning("Login failure branch entered.")

        except Exception as e:
             logging.error(f"Error during login process: {e}", exc_info=True) # Log full traceback
             print(f"An internal error occurred during login: {e}") # Show error in widget too

def on_signup_clicked(b):
    print("--- Signup Clicked ---") # Print to notebook output
    with login_output: # Ensure output goes to the widget
        clear_output(wait=True) # Clear previous messages in this widget
        logging.info("Signup button event triggered.")
        try:
            user = login_username.value
            pwd = login_password.value
            logging.info(f"Attempting signup for user: '{user}'")
            success, msg = create_account(user, pwd)
            print(msg) # Print result message inside the login_output widget

        except Exception as e:
            logging.error(f"Error during signup process: {e}", exc_info=True) # Log full traceback
            print(f"An internal error occurred during signup: {e}") # Show error in widget too

# --- Connect Callbacks ---
# Make sure this is done AFTER functions are defined
login_button.on_click(on_login_clicked)
signup_button.on_click(on_signup_clicked)

# --- UI Layout ---
# Ensure login_box and main_area are defined before use in run_prototype
login_box = widgets.VBox([
    widgets.HTML("<h3>Login or Sign Up</h3>"), # Added heading
    login_username,
    login_password,
    widgets.HBox([login_button, signup_button]),
    login_output # Make sure the output widget is part of the displayed box!
])


In [6]:
# --- Step 3: Demographic & Intake (From main_prototype - UNCHANGED) ---
def show_intake_form(username):
    intake_form_box_widgets = []
    age = widgets.IntText(description="Age:", value=30)
    gender = widgets.Dropdown(options=["Male", "Female", "Non-binary", "Other", "Prefer not to say"], description="Gender:")
    ethnicity = widgets.Text(description="Ethnicity (Optional):")
    socioeconomic = widgets.Dropdown(options=["Low", "Medium", "High", "Prefer not to say"], description="Socioeconomic Status (Optional):")
    sleep = widgets.IntSlider(value=7, min=0, max=14, description="Avg Sleep (hrs):")
    appetite = widgets.Dropdown(options=["Normal", "Decreased", "Increased", "Fluctuates"], description="Appetite Change:")
    energy = widgets.IntSlider(value=5, min=0, max=10, description="Energy Level (0-10):")
    social = widgets.Dropdown(options=["Very Active", "Moderately Active", "Somewhat Withdrawn", "Very Withdrawn"], description="Social Activity:")
    intake_form_box_widgets.extend([age, gender, ethnicity, socioeconomic, sleep, appetite, energy, social])

    submit_button = widgets.Button(description="Submit Intake")
    intake_output = widgets.Output()

    def on_submit_clicked(b):
        with main_area: # Display output in the main area
             clear_output(wait=True)
             user_db[username]["data"]["intake"] = {
                "age": age.value, "gender": gender.value, "ethnicity": ethnicity.value,
                "socioeconomic": socioeconomic.value, "sleep": sleep.value,
                "appetite": appetite.value, "energy": energy.value, "social": social.value
             }
             print("Intake data submitted.")
             simulate_classification(username) # Proceed to classification

    submit_button.on_click(on_submit_clicked)
    intake_box = widgets.VBox([*intake_form_box_widgets, submit_button, intake_output])
    with main_area:
         clear_output(wait=True)
         print(f"Welcome {username}! Please provide some optional details.")
         display(intake_box)


In [7]:
# --- Step 4: Model 4 – Mental Health Classification (Dummy - From main_prototype - UNCHANGED) ---
def train_mental_health_classifier():
    X_train = np.array([[25, 7, 6, 3],[40, 5, 4, 1],[30, 8, 7, 4],[22, 6, 5, 2], [55, 4, 3, 1], [35, 9, 8, 3]])
    y_train = np.array([0, 1, 0, 1, 1, 0]) # 0=Low risk, 1=High risk
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_train, y_train)
    return clf
mental_health_classifier = train_mental_health_classifier()

def simulate_classification(username):
    intake = user_db[username]["data"].get("intake", {})
    if not intake: print("No intake data."); return
    # Simplified feature mapping for the dummy model
    features = np.array([[
        intake.get("age", 30), intake.get("sleep", 7), intake.get("energy", 5),
        {"Very Active": 3, "Moderately Active": 2, "Somewhat Withdrawn": 1, "Very Withdrawn": 0}.get(intake.get("social"), 1)
    ]])
    risk = mental_health_classifier.predict(features)[0]
    prob = mental_health_classifier.predict_proba(features)[0][risk]
    issues = {"Depression": 0.0, "Anxiety": 0.0, "Stress & Burnout": 0.0}
    if risk == 1:
        issues["Depression"] = round(min(1.0, prob + random.uniform(0.1, 0.4)), 2)
        issues["Anxiety"] = round(min(1.0, prob + random.uniform(0.2, 0.5)), 2)
        issues["Stress & Burnout"] = round(min(1.0, prob + random.uniform(0.3, 0.6)), 2)
    else:
        issues["Depression"] = round(max(0.0, prob - random.uniform(0.1, 0.3)), 2)
        issues["Anxiety"] = round(max(0.0, prob - random.uniform(0.2, 0.4)), 2)
        issues["Stress & Burnout"] = round(max(0.0, prob - random.uniform(0.1, 0.2)), 2)

    user_db[username]["data"]["issues"] = issues
    print("\nBased on intake data (experimental prediction):")
    for issue, p in issues.items(): print(f" - {issue}: {int(p * 100)}% likelihood")
    show_chat_mode_selection(username) # Proceed to chat mode selection



In [9]:
# --- (Steps 2, 3, 4: Auth, Intake, Dummy Classification - UNCHANGED from previous) ---
# Keep the code for user_db, create_account, login, show_intake_form,
# train_mental_health_classifier, simulate_classification here...
# --- BEGIN Steps 2,3,4 ---
user_db = {} # Simulated user database

def create_account(username, password):
    if username in user_db: return False, "Username exists."
    user_db[username] = {"password": password, "data": {}}
    return True, "Account created."

def login(username, password):
    if username in user_db and user_db[username]["password"] == password: return True, "Login successful."
    return False, "Invalid credentials."

# Widgets for login
login_username = widgets.Text(description="Username:")
login_password = widgets.Password(description="Password:")
login_button = widgets.Button(description="Login")
signup_button = widgets.Button(description="Sign Up")
login_output = widgets.Output()
main_area = widgets.Output() # Area to display next steps after login

current_username = None # Store logged in user

def on_login_clicked(b):
    global current_username
    with login_output:
        clear_output(wait=True)
        user = login_username.value; pwd = login_password.value
        success, msg = login(user, pwd)
        print(msg)
        if success:
            current_username = user
            login_box.layout.display = 'none' # Hide login
            with main_area:
                clear_output(wait=True)
                show_intake_form(user)

def on_signup_clicked(b):
    with login_output:
        clear_output(wait=True)
        user = login_username.value; pwd = login_password.value
        success, msg = create_account(user, pwd)
        print(msg)

login_button.on_click(on_login_clicked)
signup_button.on_click(on_signup_clicked)
login_box = widgets.VBox([login_username, login_password, widgets.HBox([login_button, signup_button]), login_output])


# Step 3: Demographic & Intake (From main_prototype - UNCHANGED)
def show_intake_form(username):
    intake_form_box_widgets = []
    age = widgets.IntText(description="Age:", value=30)
    gender = widgets.Dropdown(options=["Male", "Female", "Non-binary", "Other", "Prefer not to say"], description="Gender:")
    ethnicity = widgets.Text(description="Ethnicity (Optional):")
    socioeconomic = widgets.Dropdown(options=["Low", "Medium", "High", "Prefer not to say"], description="Socioeconomic Status (Optional):")
    sleep = widgets.IntSlider(value=7, min=0, max=14, description="Avg Sleep (hrs):")
    appetite = widgets.Dropdown(options=["Normal", "Decreased", "Increased", "Fluctuates"], description="Appetite Change:")
    energy = widgets.IntSlider(value=5, min=0, max=10, description="Energy Level (0-10):")
    social = widgets.Dropdown(options=["Very Active", "Moderately Active", "Somewhat Withdrawn", "Very Withdrawn"], description="Social Activity:")
    intake_form_box_widgets.extend([age, gender, ethnicity, socioeconomic, sleep, appetite, energy, social])

    submit_button = widgets.Button(description="Submit Intake")
    intake_output = widgets.Output()

    def on_submit_clicked(b):
        with main_area: # Display output in the main area
             clear_output(wait=True)
             user_db[username]["data"]["intake"] = {
                "age": age.value, "gender": gender.value, "ethnicity": ethnicity.value,
                "socioeconomic": socioeconomic.value, "sleep": sleep.value,
                "appetite": appetite.value, "energy": energy.value, "social": social.value
             }
             print("Intake data submitted.")
             simulate_classification(username) # Proceed to classification

    submit_button.on_click(on_submit_clicked)
    intake_box = widgets.VBox([*intake_form_box_widgets, submit_button, intake_output])
    with main_area:
         clear_output(wait=True)
         print(f"Welcome {username}! Please provide some optional details.")
         display(intake_box)

# Step 4: Model 4 – Mental Health Classification (Dummy - From main_prototype - UNCHANGED)
def train_mental_health_classifier():
    X_train = np.array([[25, 7, 6, 3],[40, 5, 4, 1],[30, 8, 7, 4],[22, 6, 5, 2], [55, 4, 3, 1], [35, 9, 8, 3]])
    y_train = np.array([0, 1, 0, 1, 1, 0]) # 0=Low risk, 1=High risk
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_train, y_train)
    return clf
mental_health_classifier = train_mental_health_classifier()

def simulate_classification(username):
    intake = user_db[username]["data"].get("intake", {})
    if not intake: print("No intake data."); return
    # Simplified feature mapping for the dummy model
    features = np.array([[
        intake.get("age", 30), intake.get("sleep", 7), intake.get("energy", 5),
        {"Very Active": 3, "Moderately Active": 2, "Somewhat Withdrawn": 1, "Very Withdrawn": 0}.get(intake.get("social"), 1)
    ]])
    risk = mental_health_classifier.predict(features)[0]
    prob = mental_health_classifier.predict_proba(features)[0][risk]
    issues = {"Depression": 0.0, "Anxiety": 0.0, "Stress & Burnout": 0.0}
    if risk == 1:
        issues["Depression"] = round(min(1.0, prob + random.uniform(0.1, 0.4)), 2)
        issues["Anxiety"] = round(min(1.0, prob + random.uniform(0.2, 0.5)), 2)
        issues["Stress & Burnout"] = round(min(1.0, prob + random.uniform(0.3, 0.6)), 2)
    else:
        issues["Depression"] = round(max(0.0, prob - random.uniform(0.1, 0.3)), 2)
        issues["Anxiety"] = round(max(0.0, prob - random.uniform(0.2, 0.4)), 2)
        issues["Stress & Burnout"] = round(max(0.0, prob - random.uniform(0.1, 0.2)), 2)

    user_db[username]["data"]["issues"] = issues
    print("\nBased on intake data (experimental prediction):")
    for issue, p in issues.items(): print(f" - {issue}: {int(p * 100)}% likelihood")
    show_chat_mode_selection(username) # Proceed to chat mode selection
# --- END Steps 2,3,4 ---



In [ ]:


# --- Step 5: Load Models ---

# A. Face Emotion Model (Dummy - From main_prototype) - Not used in voice focus
# def build_face_emotion_model(...): ... (Assume it's defined as before)
# face_emotion_model = build_face_emotion_model()

# B. Voice Emotion Model (Load pre-trained Keras LSTM from SER example)
SER_MODEL_PATH = 'speech_emotion_model.h5' # MAKE SURE this file exists from your SER training
ser_model = None
ser_encoder = None
SER_EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad'] # Match training labels

try:
    # Create and fit the encoder based on known labels
    ser_encoder = OneHotEncoder()
    ser_encoder.fit(np.array(SER_EMOTIONS).reshape(-1, 1))
    print("SER Label Encoder created.")

    # Load the pre-trained Keras model
    ser_model = load_model(SER_MODEL_PATH)
    print(f"Loaded SER model from {SER_MODEL_PATH}")
    ser_model.summary() # Print model structure
except Exception as e:
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print(f"ERROR loading SER model from {SER_MODEL_PATH}: {e}")
    print(f"Voice Emotion Analysis will NOT work. Make sure the model file exists")
    print(f"and is compatible with your TensorFlow/Keras version.")
    print(f"You might need to retrain it using the code from the SER example")
    print(f"in this environment and save it as {SER_MODEL_PATH}")
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    ser_model = None # Ensure it's None if loading failed

def analyze_voice_emotion_keras(audio_chunk):
    """Analyzes a chunk of audio using the Keras SER model."""
    if ser_model is None or ser_encoder is None:
        return "SER Model Error", 0.0

    emotion = "unknown"
    confidence = 0.0
    try:
        # Ensure chunk is float32 numpy array
        if audio_chunk.dtype != np.float32:
            audio_chunk = audio_chunk.astype(np.float32) / np.iinfo(audio_chunk.dtype).max # Normalize if needed

        # Extract MFCCs (adjust duration/offset if needed for streaming)
        # Need enough audio for MFCC - might need to buffer chunks
        # For now, assume chunk is long enough (e.g., AUDIO_CHUNK_DURATION >= 0.5s)
        if len(audio_chunk) < 2048: # Need minimum length for librosa processing
             return "Chunk too short", 0.0

        # Use y=audio_chunk directly, set duration matching training feature extraction?
        mfcc = np.mean(librosa.feature.mfcc(y=audio_chunk, sr=AUDIO_SAMPLE_RATE, n_mfcc=40).T, axis=0)

        # Reshape for LSTM model (samples, timesteps, features) -> needs (1, 40, 1) for your model
        mfcc_reshaped = np.expand_dims(mfcc, axis=0) # Add batch dimension -> (1, 40)
        mfcc_reshaped = np.expand_dims(mfcc_reshaped, axis=-1) # Add feature dimension -> (1, 40, 1)

        # Predict using the Keras model
        prediction = ser_model.predict(mfcc_reshaped, verbose=0) # verbose=0 prevents print spam

        # Decode prediction
        emotion_index = np.argmax(prediction[0])
        confidence = prediction[0][emotion_index]
        # Inverse transform the one-hot encoded index back to label
        emotion = ser_encoder.inverse_transform(np.array(emotion_index).reshape(1, -1))[0][0]

    except Exception as e:
        print(f"SER Prediction Error: {e}")
        emotion = "SER Predict Error"
        confidence = 0.0

    return emotion, confidence

# C. NLP Text Analysis Pipeline (Use Hugging Face - requires transformers & torch/tf)
print("Loading NLP pipeline...")
nlp_text_analyzer = None
try:
    nlp_text_analyzer = pipeline("sentiment-analysis") # Or another like "text-classification" for emotion
    print("NLP pipeline loaded.")
except Exception as e:
    print(f"Warning: Failed to load NLP pipeline: {e}. Text analysis will be basic.")

def analyze_text_nlp_hf(text):
     if nlp_text_analyzer:
         try:
             result = nlp_text_analyzer(text)[0]
             return result["label"], result["score"] # e.g., POSITIVE, 0.98
         except Exception as e:
             print(f"NLP analysis failed: {e}")
             return "NLP Error", 0.0
     else:
         # Basic fallback if pipeline failed to load
         if "sad" in text.lower() or "down" in text.lower(): return "NEGATIVE", 0.7
         if "happy" in text.lower() or "good" in text.lower(): return "POSITIVE", 0.7
         return "NEUTRAL", 0.5

# D. LLM Setup (Gemini)
print("Configuring LLM...")
llm_model = None
try:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
    if GOOGLE_API_KEY:
        genai.configure(api_key=GOOGLE_API_KEY)
        llm_model = genai.GenerativeModel('gemini-pro')
        print("Google Gemini configured.")
    else:
        print("Warning: GOOGLE_API_KEY environment variable not found. LLM disabled.")
except Exception as e:
    print(f"Error configuring Google Gemini: {e}. LLM disabled.")

# E. TTS Setup
print("Initializing TTS engine...")
tts_engine = None
try:
    tts_engine = pyttsx3.init()
    print("TTS engine initialized.")
except Exception as e:
    print(f"Warning: Could not initialize TTS engine: {e}. Speech output disabled.")


# --- Step 6: Real-time Processing Threads & Functions ---

# A. Audio Input Stream Callback
audio_stream = None
def audio_callback(indata, frames, time_info, status):
    """Called by sounddevice for each new audio chunk."""
    if status:
        print("Audio Stream Status:", status)
    if not stop_event.is_set():
        try:
            # Put a copy onto the queue for SER/Spectrogram processing
            audio_chunk_q.put(indata.copy(), timeout=0.1)
        except queue.Full:
            pass # Ignore if the queue is full, skip this chunk


# B. Audio Processing Thread (SER Model + Spectrogram)
def audio_processing_thread_ser():
    global ser_fig, ser_ax # Need to modify global plot objects

    print("Audio SER/Spectrogram thread started.")
    local_buffer = np.array([], dtype=np.float32)

    while not stop_event.is_set():
        try:
            # Get audio chunk from the queue
            audio_chunk = audio_chunk_q.get(timeout=0.5) # Wait up to 0.5s

            # Ensure it's a 1D array for processing
            if audio_chunk.ndim > 1:
                audio_chunk = audio_chunk.flatten()

             # ---- Run SER Model ----
            voice_emotion, ser_confidence = analyze_voice_emotion_keras(audio_chunk)
            if voice_emotion not in ["Chunk too short", "SER Model Error", "SER Predict Error"]:
                 timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                 with voice_emotion_log_lock:
                     voice_emotion_log.append((timestamp, voice_emotion, ser_confidence))
                 # print(f"Time: {timestamp}, Voice Emotion: {voice_emotion} ({ser_confidence:.2f})") # Debug print

            # ---- Update Spectrogram ----
            if ser_fig is not None and ser_ax is not None and spectrogram_output_widget is not None:
                 try:
                     # Calculate Mel Spectrogram
                     S = librosa.feature.melspectrogram(y=audio_chunk.astype(np.float32), sr=AUDIO_SAMPLE_RATE, n_mels=128, fmax=8000)
                     S_dB = librosa.power_to_db(S, ref=np.max)

                     # Update plot data
                     ser_ax.clear() # Clear previous plot content
                     img = librosa.display.specshow(S_dB, sr=AUDIO_SAMPLE_RATE, x_axis='time', y_axis='mel', ax=ser_ax, fmax=8000)
                     # ser_fig.colorbar(img, ax=ser_ax, format='%+2.0f dB') # Adding colorbar repeatedly is tricky/slow in widget updates
                     ser_ax.set_title(f"Live Spectrogram (Emotion: {voice_emotion})")

                     # Update the display within the widget
                     with spectrogram_output_widget:
                         clear_output(wait=True) # Clear the previous plot in the widget
                         display(ser_fig) # Display the updated figure

                 except Exception as plot_e:
                      print(f"Spectrogram plotting error: {plot_e}")

            # Clean queue task (important!)
            audio_chunk_q.task_done()

        except queue.Empty:
            continue # No new audio chunk, loop again
        except Exception as thread_e:
             print(f"Error in SER/Spectrogram Thread: {thread_e}")
             time.sleep(0.1) # Avoid busy-looping on persistent errors

    print("Audio SER/Spectrogram thread finished.")

# C. Speech-to-Text Thread (Using SpeechRecognition - Largely same as before)
stt_recognizer = sr.Recognizer()
def speech_to_text_thread():
    print("STT thread started.")
    # Use default microphone as detected by SpeechRecognition
    mic = sr.Microphone(sample_rate=AUDIO_SAMPLE_RATE) # Ensure sample rate match if needed by backend
    # Set energy threshold dynamically once
    with mic as source:
        print("STT: Adjusting for ambient noise...")
        try:
           stt_recognizer.adjust_for_ambient_noise(source, duration=1.0)
           print(f"STT: Ambient noise energy threshold set to {stt_recognizer.energy_threshold:.2f}")
        except Exception as e:
             print(f"STT: Ambient noise adjustment failed: {e}. Using default.")

    while not stop_event.is_set():
        if listen_event.wait(timeout=1.0): # Wait for main loop to signal listening
            print("\n>>> STT Listening... (Speak Now!) <<<")
            text = "[STT No Trigger]"
            try:
                with mic as source:
                    # listen blocks this thread, but that's its job
                    audio_data = stt_recognizer.listen(source, phrase_time_limit=STT_PHRASE_TIME_LIMIT, timeout=STT_PHRASE_TIME_LIMIT + 1) # Added timeout
                print(">>> STT Processing... <<<")

                # Convert AudioData to a format Pydub can handle (like WAV bytes)
                wav_bytes = audio_data.get_wav_data()
                # Alternative: Use recognize_google directly if it works
                text = stt_recognizer.recognize_google(audio_data)

            except sr.WaitTimeoutError: text = "[STT Timeout]"
            except sr.UnknownValueError: text = "[STT Audio Not Understood]"
            except sr.RequestError as e: text = f"[STT API Error: {e}]"
            except Exception as e: text = f"[STT System Error: {e}]"
            finally:
                if text.startswith("["): print(text) # Print errors/status immediately
                stt_text_q.put(text) # Put result onto queue
                listen_event.clear() # Reset listening trigger

    print("STT thread finished.")


# D. Text-to-Speech Thread (Same as before)
def text_to_speech_thread():
    print("TTS thread started.")
    while not stop_event.is_set():
        try:
            text = tts_q.get(timeout=0.5)
            if tts_engine:
                # print(f"[TTS Speaking]: {text}") # Reduce console spam
                tts_engine.say(text)
                tts_engine.runAndWait()
            # else: print(f"[TTS Disabled]: {text}") # Debugging
            tts_q.task_done()
        except queue.Empty:
            continue
        except Exception as e:
            print(f"Error in TTS thread: {e}")
            if not tts_q.empty():
                try: tts_q.task_done()
                except ValueError: pass
    print("TTS thread finished.")


# --- Step 7: Chatbot Logic Function (Integrates LLM if available) ---
def get_chatbot_reply_final(user_text, current_voice_emotion, fallback_face_emotion="N/A"):
    """Generates chatbot reply using NLP, context, and LLM or fallback."""
    global chat_history
    bot_response = "..."

    # Handle STT errors first
    if user_text.startswith("[STT") and user_text.endswith("]"):
        if "Not Understood" in user_text: return random.choice(["Sorry, say again?", "Didn't catch that.", "My audio input missed that."])
        elif "Timeout" in user_text: return random.choice(["Didn't hear anything.", "Were you speaking?", "Listening timed out."])
        else: return "Hmm, technical glitch with speech recognition."

    # Get text analysis
    text_sentiment, _ = analyze_text_nlp_hf(user_text)
    # Note: We have live *voice* emotion, so maybe text emotion is less critical now?
    print(f"[NLP Analysis]: Text Sentiment='{text_sentiment}'") # Log it anyway

    # Add user message to history
    with chat_history_lock:
        chat_history.append({'role': 'user', 'parts': [user_text]})
        limited_history = chat_history[-10:] # Keep last 5 turns

    # Try LLM if available
    if llm_model:
        try:
            context_prompt = f"""You are an empathetic AI assistant. Respond kindly. Keep responses concise. If the user mentions severe distress or self-harm, gently advise contacting a crisis hotline (like 988) or professional immediately.

            Current user live voice emotion detected: {current_voice_emotion}
            User text sentiment: {text_sentiment}

            Conversation History:""" # History will be appended

            # Prepare content for Gemini API
            gemini_content = [{'role': 'user', 'parts': [context_prompt]},
                              {'role': 'model', 'parts': ["Okay, understood. How can I help based on the latest message and context?"]}]
            gemini_content.extend(limited_history)

            response = llm_model.generate_content(gemini_content, stream=False,
                                                   # Add safety settings if concerned about refusals
                                                    safety_settings={'HARASSMENT':'block_none',
                                                                      'HATE_SPEECH': 'block_none',
                                                                      'SEXUALLY_EXPLICIT':'block_none',
                                                                      'DANGEROUS_CONTENT':'block_none'})

            if response.parts:
                bot_response = response.text
            else:
                 # Handle potential blocks despite safety settings
                 print(f"Warning: LLM response empty/blocked. Feedback: {response.prompt_feedback}")
                 bot_response = "I'm currently unable to respond to that specific topic. Could we discuss something else?"
        except Exception as e:
            print(f"LLM Error: {e}")
            bot_response = random.choice(["My AI brain stumbled. Try again?", "Connection issue with my core logic."]) # Fallback on LLM error
    else:
        # LLM Disabled - Very Basic Fallback (could be improved)
        if "hello" in user_text.lower(): bot_response = "Hi there!"
        elif "sad" in user_text.lower(): bot_response = f"I hear sadness. Voice emotion detected: {current_voice_emotion}. How can I help?"
        elif "happy" in user_text.lower(): bot_response = f"Glad things seem happy! Voice emotion detected: {current_voice_emotion}."
        else: bot_response = f"Okay. Current voice emotion: {current_voice_emotion}. Tell me more."

    # Add bot response to history
    with chat_history_lock:
        chat_history.append({'role': 'model', 'parts': [bot_response]})
        # Prune history
        if len(chat_history) > 40: chat_history = chat_history[-10:] # Keep it manageable


    # Check for exit command
    if "goodbye quit now" in user_text.lower():
        bot_response = "Okay, ending session. Take care!"
        # Signal stop cleanly AFTER sending the final response to TTS
        tts_q.put(bot_response)
        tts_q.join() # Wait for TTS to finish this last message
        stop_event.set()
        print("Exit command recognized.") # Debug print

    return bot_response


# --- Step 8: Chat Interface Logic ---

def show_chat_mode_selection(username):
    chat_mode = widgets.Dropdown(options=["Text Chat", "Voice Chat", "Video Chat"], description="Chat Mode:")
    start_chat_button = widgets.Button(description="Start Chat")
    chat_mode_output = widgets.Output()

    def on_start_chat(b):
        with chat_mode_output:
            clear_output(wait=True)
            mode = chat_mode.value
            print(f"Starting {mode} session...")

            # Reset global states for new session
            global chat_history, voice_emotion_log, stop_event, ser_fig, ser_ax, spectrogram_output_widget
            stop_event.clear() # Ensure stop event is reset
            chat_history = []
            voice_emotion_log = []
            ser_fig, ser_ax = None, None # Reset plot figure
            spectrogram_output_widget = None


            # Disable the mode selection widgets after starting
            chat_mode.disabled = True
            start_chat_button.disabled = True

            if mode == "Text Chat":
                text_chat_session(username)
            elif mode == "Voice Chat":
                voice_chat_session(username) # Launch the enhanced voice chat
            elif mode == "Video Chat":
                # video_chat_session(username) # Need to integrate video aspects from previous step if wanted
                 print("Video chat requires further integration (using OpenCV/DeepFace). Falling back to Text chat for now.")
                 text_chat_session(username)

    start_chat_button.on_click(on_start_chat)

    with main_area:
        clear_output(wait=True)
        print("\nPlease select your preferred chat mode:")
        display(widgets.VBox([chat_mode, start_chat_button, chat_mode_output]))


# A. Text Chat Session (Largely unchanged from main_prototype, uses NLP)
def text_chat_session(username):
    global chat_history # Use global history

    text_input = widgets.Textarea(placeholder="Type your message...", description="You:")
    send_button = widgets.Button(description="Send")
    chat_log_output = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'scroll'})
    end_chat_button = widgets.Button(description="End Text Chat & View Report")


    # Display initial message
    with chat_log_output:
         clear_output()
         initial_msg = "Bot: You've chosen text chat. How can I help you today?"
         print(initial_msg)
         chat_history.append({'role': 'model', 'parts': [initial_msg]})


    def on_send_clicked(b):
        user_msg = text_input.value
        if not user_msg: return
        text_input.value = ""

        with chat_log_output: print(f"\nYou: {user_msg}")

        # --- Get bot response (Use enhanced function, voice emotion N/A here) ---
        # For text chat, facial emotion might be relevant if video was also active, but voice isn't.
        bot_reply = get_chatbot_reply_final(user_msg, current_voice_emotion="N/A")
        with chat_log_output: print(f"Bot: {bot_reply}")
        # No TTS for pure text chat? Or add optional TTS? Add for consistency maybe.
        if tts_engine: tts_q.put(bot_reply)

        if "Okay, ending session." in bot_reply: # Check if bot decided to end
            end_chat_procedure(username)

    def on_end_chat_clicked(b):
        end_chat_procedure(username)

    send_button.on_click(on_send_clicked)
    end_chat_button.on_click(on_end_chat_clicked)

    text_chat_box = widgets.VBox([chat_log_output, text_input, send_button, end_chat_button])
    with main_area:
        clear_output(wait=True)
        print("--- Text Chat Session ---")
        display(text_chat_box)

# B. Enhanced Voice Chat Session (Integrates Live SER/Spectrogram, STT, TTS, LLM)
def voice_chat_session(username):
    global chat_history, voice_emotion_log, audio_stream, stop_event, ser_fig, ser_ax, spectrogram_output_widget

    # --- UI Setup ---
    # Area for the spectrogram plot
    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc', 'margin_bottom': '10px'})
    # Area for the conversation text log
    chat_log_output_vc = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'scroll'})
    # Button to trigger listening (instead of typing) - though STT thread listens after pauses too
    listen_button = widgets.Button(description="🎤 Start Speaking (Press & Hold or Tap)", button_style='info', icon='microphone')
    # Button to end the session
    end_chat_button_vc = widgets.Button(description="End Voice Chat & View Report", button_style='danger')
    status_label = widgets.Label(value="Status: Initializing...")

    # Initialize the plot figure and axes
    ser_fig, ser_ax = plt.subplots(figsize=(7, 4)) # Slightly larger fig size
    ser_ax.set_title("Live Spectrogram - Initializing...")
    plt.close(ser_fig) # Close the initial display, we'll show it in the widget

    # Arrange UI
    voice_chat_box = widgets.VBox([
        status_label,
        spectrogram_output_widget,
        chat_log_output_vc,
        widgets.HBox([listen_button, end_chat_button_vc]) # Horizontal buttons
    ])

    # Display the initial UI
    with main_area:
        clear_output(wait=True)
        print("--- Voice Chat Session ---")
        # Display placeholder plot initially
        with spectrogram_output_widget:
            display(ser_fig)
        display(voice_chat_box)


    # --- Start Threads & Audio Stream ---
    status_label.value = "Status: Starting threads..."
    print("Starting threads for voice session...")
    # Ensure previous threads are stopped if any lingering
    stop_event.clear()

    # Reset logs for this session
    chat_history = []
    voice_emotion_log = []

    # Start necessary threads
    threads = []
    stt_thread = threading.Thread(target=speech_to_text_thread)
    threads.append(stt_thread)
    stt_thread.start()

    ser_thread = threading.Thread(target=audio_processing_thread_ser)
    threads.append(ser_thread)
    ser_thread.start()

    if tts_engine:
        tts_thread = threading.Thread(target=text_to_speech_thread)
        threads.append(tts_thread)
        tts_thread.start()

    try:
        status_label.value = "Status: Starting audio stream..."
        # Start the audio stream
        audio_stream = sd.InputStream(
            samplerate=AUDIO_SAMPLE_RATE,
            channels=AUDIO_CHANNELS,
            blocksize=AUDIO_CHUNK_SAMPLES, # Process chunks of this size
            callback=audio_callback,
            dtype='float32' # Use float32 for easier processing
        )
        audio_stream.start()
        status_label.value = "Status: Ready. Press 'Start Speaking' or just talk."
        print("Audio stream started.")

        # Initial greeting
        initial_msg = "Bot: Voice chat activated. How can I help?"
        with chat_log_output_vc: print(initial_msg)
        if tts_engine: tts_q.put(initial_msg)
        chat_history.append({'role': 'model', 'parts': [initial_msg]})

    except Exception as e:
        status_label.value = f"Status: ERROR starting audio stream: {e}"
        print(f"Fatal Error starting audio stream: {e}")
        stop_event.set() # Stop threads if stream fails
        # Join threads?
        return # Exit function


    # --- Button Actions ---
    def on_listen_button_clicked(b):
         if not listen_event.is_set():
              listen_event.set() # Signal STT thread to start immediate listen cycle
              status_label.value = "Status: Listening actively..."
              listen_button.description = "Listening... (Tap again?)" # Change button text
              listen_button.button_style = 'warning'
         else:
              # Maybe force stop listening? Hard with current SR listen structure
              status_label.value = "Status: Already listening or processing..."


    def on_end_vc_clicked(b):
        status_label.value = "Status: Ending Session..."
        stop_event.set() # Signal stop
        listen_button.disabled = True
        end_chat_button_vc.disabled = True
        # Cleanup will happen in main loop exit or finally block


    listen_button.on_click(on_listen_button_clicked)
    end_chat_button_vc.on_click(on_end_vc_clicked)


    # --- Main Interaction Loop (Checks STT Queue) ---
    # This loop doesn't block UI updates because threads do the work
    # It just checks the queue periodically
    # Need to run this loop *outside* the voice_chat_session function
    # or start it in a *separate management thread* to allow
    # the function to return and the main notebook cell to finish setup.

    # --> Let's move the check loop to the main execution part after launching voice_chat_session


# C. Video Chat Session (Placeholder - needs integration)
def video_chat_session(username):
    print("Video chat session needs implementation (OpenCV, DeepFace integration).")
    # Fallback to voice or text for now
    voice_chat_session(username)


# --- Step 9: Session Report Generation ---
def generate_report(username):
    data = user_db[username]["data"]
    report_lines = []
    report_lines.append("="*40)
    report_lines.append("  Multi-Modal Mental Health Chatbot Session Report")
    report_lines.append("="*40)
    report_lines.append(f"User: {username}")
    report_lines.append(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    report_lines.append("--- User Intake Summary ---")
    if data.get("intake"):
        for key, value in data["intake"].items(): report_lines.append(f"  {key.capitalize()}: {value}")
    else: report_lines.append("  (No intake data provided)")

    report_lines.append("\n--- Initial Assessment (Experimental) ---")
    if data.get("issues"):
        for issue, prob in data["issues"].items(): report_lines.append(f"  Potential for {issue}: {int(prob * 100)}% likelihood")
    else: report_lines.append("  (No assessment data)")

    # --- Add Voice Emotion Log ---
    report_lines.append("\n--- Voice Emotion Log (During Voice Chat) ---")
    if voice_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence")
         report_lines.append("  ------------|-----------------|-----------")
         # Lock access while iterating/reading
         with voice_emotion_log_lock:
              # Limit reported entries if log is very long
              log_limit = 50
              display_log = voice_emotion_log[-log_limit:] if len(voice_emotion_log) > log_limit else voice_emotion_log
              if len(voice_emotion_log) > log_limit: report_lines.append("  ...(trimmed to last 50 entries)...")

              for timestamp, emotion, confidence in display_log:
                   report_lines.append(f"  {timestamp} | {emotion:<15} | {confidence:.2f}")
    else:
         report_lines.append("  (No voice chat session or no voice emotions logged)")
    # ---------------------------

    report_lines.append("\n--- Conversation Transcript ---")
    if chat_history:
        for turn in chat_history:
            speaker = turn.get('role', 'unknown').capitalize()
            text = turn.get('parts', [""])[0] # Get first part
            report_lines.append(f"  {speaker}: {text}")
    else: report_lines.append("  (No conversation recorded)")

    report_lines.append("\n"+"="*40)
    report_content = "\n".join(report_lines)
    filename = f"{username}_session_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

    try:
        with open(filename, "w", encoding='utf-8') as f:
            f.write(report_content)
        print(f"\nSession report generated successfully: '{filename}'")
        print("You can open this text file to review your session.")
    except Exception as e:
        print(f"\nError writing report file: {e}")

    # --- Professional Help Suggestion ---
    issues = data.get("issues", {})
    high_risk = any(prob > 0.65 for prob in issues.values()) # Adjust threshold
    if high_risk:
         print("\nRecommendation: Based on the session, consider reaching out to a mental health professional.")
         print("Resources: Crisis Text Line, 988 Lifeline (US/Canada), Befrienders Worldwide.")
         # In a real app: Link to find resources.
    else:
         print("\nRecommendation: Session indicates moderate or low distress. Continue self-care practices.")


def end_chat_procedure(username):
    """Stops threads, streams and generates the report."""
    print("\nEnding chat session and generating report...")
    # 1. Signal all threads to stop
    stop_event.set()

    # 2. Stop the audio stream if it's running
    global audio_stream
    if audio_stream is not None and audio_stream.active:
        try:
            print("Stopping audio stream...")
            audio_stream.stop()
            audio_stream.close()
            print("Audio stream stopped.")
        except Exception as e:
            print(f"Error stopping audio stream: {e}")
        finally:
             audio_stream = None # Clear reference

    # 3. Wait for threads to finish (optional, but good practice)
    # Note: Joining daemon threads might not be strictly needed but helps ensure cleanup
    print("Waiting briefly for threads to exit...")
    time.sleep(1.0) # Give threads a moment to react to stop_event

    # 4. Generate Report
    with main_area: # Display report info in main area
        clear_output(wait=True)
        generate_report(username)
        # Offer option to login again or exit?
        print("\nSession ended. You can close the application or run the first cell again to log in.")
        # Consider re-displaying the login box?
        # login_box.layout.display = 'flex'


# --- Step 10: Main Application Execution Logic ---
def run_prototype():
    clear_output()
    print("="*50)
    print(" Multi-Modal Mental Health Chatbot Prototype")
    print("="*50)
    display(login_box)
    display(main_area)

    # --- Interaction Loop ---
    # This loop now primarily just keeps the notebook cell running
    # while the background threads handle interactions triggered by buttons/voice.
    # It also checks the STT queue to process user speech input.
    try:
         print("\nPrototype running. Waiting for user interaction...")
         while not stop_event.is_set(): # Keep running until explicitly stopped
            try:
                 # Check for transcribed text from STT thread
                 user_text = stt_text_q.get(timeout=0.5) # Check every 0.5 seconds

                 # Find the active chat output widget to print to
                 # This is a bit hacky - assumes voice chat is active if it was started
                 # Need better state management for active session type
                 current_chat_output = None
                 try:
                     # Attempt to access voice chat output if it exists
                     # Accessing widgets created inside functions needs care
                     # We need a better way to pass widget references or use a class structure
                      pass # TODO: Find a reliable way to get the right output widget

                 except NameError:
                      # Likely in text chat or no chat started
                      pass


                 # --- Process the Text ---
                 if user_text:
                     print(f"\n[STT Received]: {user_text}") # Debug
                     with main_area: # Fallback to main area if specific widget not found
                          # This assumes 'current_username' is set
                          print(f"You (voice): {user_text}")

                     # Get current voice emotion (best effort)
                     current_voice_emotion = "N/A"
                     if voice_emotion_log: # Check if log has entries
                         with voice_emotion_log_lock:
                             # Get the emotion from the latest log entry
                             if voice_emotion_log: # Double check inside lock
                                 current_voice_emotion = voice_emotion_log[-1][1]


                     # Get bot reply
                     bot_reply = get_chatbot_reply_final(user_text, current_voice_emotion)

                     with main_area: # Fallback display
                         print(f"Bot: {bot_reply}")
                     if tts_engine: tts_q.put(bot_reply) # Speak the reply

                 stt_text_q.task_done()

            except queue.Empty:
                 # No text from STT, just loop and wait
                 time.sleep(0.1) # Small sleep to prevent busy-waiting excessively
                 continue
            except Exception as main_loop_e:
                 print(f"!! Error in main interaction loop: {main_loop_e} !!")
                 time.sleep(1) # Pause briefly on error

    except KeyboardInterrupt:
        print("\nKeyboard Interrupt detected! Stopping application...")
        stop_event.set()
    finally:
         print("Main loop exited.")
         # Ensure cleanup happens even if loop breaks unexpectedly
         if not stop_event.is_set(): # If stop wasn't triggered by normal means
            stop_event.set()
            end_chat_procedure(current_username if current_username else "unknown_user") # Attempt cleanup/report

         print("Prototype Shutdown Complete.")


# --- Start the Prototype ---
run_prototype()

In [ ]:
# --- Step 5: Load Models ---
# A. Face Emotion Model (We'll use DeepFace analyze directly, no separate Keras model needed unless customizing)
FACE_EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"] # Standard DeepFace output

# B. Voice Emotion Model (Load Keras LSTM)
SER_MODEL_PATH = 'speech_emotion_model.h5'
ser_model = None; ser_encoder = None
SER_EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad'] # Match TESS labels used for training
try:
    ser_encoder = OneHotEncoder(); ser_encoder.fit(np.array(SER_EMOTIONS).reshape(-1, 1))
    ser_model = load_model(SER_MODEL_PATH)
    print(f"Loaded SER model from {SER_MODEL_PATH}")
    # ser_model.summary()
except Exception as e:
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print(f"ERROR loading SER model from {SER_MODEL_PATH}: {e}")
    print(f"Voice Emotion Analysis will NOT work. Make sure the model file exists.")
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    ser_model = None
# (Keep analyze_voice_emotion_keras function as defined before)
def analyze_voice_emotion_keras(audio_chunk):
    if ser_model is None or ser_encoder is None: return "SER Model Error", 0.0
    emotion, confidence = "unknown", 0.0
    try:
        if audio_chunk.dtype != np.float32: audio_chunk = audio_chunk.astype(np.float32) / 32767.0 # Normalize INT16 etc.
        if len(audio_chunk) < 2048: return "Chunk too short", 0.0
        mfcc = np.mean(librosa.feature.mfcc(y=audio_chunk, sr=AUDIO_SAMPLE_RATE, n_mfcc=40).T, axis=0)
        mfcc_reshaped = np.expand_dims(np.expand_dims(mfcc, axis=0), axis=-1)
        prediction = ser_model.predict(mfcc_reshaped, verbose=0)
        emotion_index = np.argmax(prediction[0])
        confidence = prediction[0][emotion_index]
        emotion = ser_encoder.inverse_transform(np.array(emotion_index).reshape(1, -1))[0][0]
    except Exception as e: print(f"SER Prediction Error: {e}"); emotion = "SER Predict Error"
    return emotion, confidence


# C. NLP Text Analysis Pipeline (Same as before)
print("Loading NLP pipeline...")
nlp_text_analyzer = None
try:
    nlp_text_analyzer = pipeline("sentiment-analysis")
    print("NLP pipeline loaded.")
except Exception as e: print(f"Warning: Failed to load NLP pipeline: {e}.")
# (Keep analyze_text_nlp_hf function as defined before)
def analyze_text_nlp_hf(text):
     if nlp_text_analyzer:
         try: result = nlp_text_analyzer(text)[0]; return result["label"], result["score"]
         except Exception as e: print(f"NLP analysis failed: {e}"); return "NLP Error", 0.0
     else: # Basic fallback
         if "sad" in text.lower(): return "NEGATIVE", 0.7
         if "happy" in text.lower(): return "POSITIVE", 0.7
         return "NEUTRAL", 0.5

# D. LLM Setup (Gemini - same as before)
print("Configuring LLM...")
llm_model = None
try: # Ensure GOOGLE_API_KEY is set as environment variable BEFORE launching Jupyter
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
    if GOOGLE_API_KEY:
        genai.configure(api_key=GOOGLE_API_KEY)
        llm_model = genai.GenerativeModel('gemini-pro') # Or 'gemini-1.5-flash'
        print("Google Gemini configured.")
    else: print("Warning: GOOGLE_API_KEY environment variable not found. LLM disabled.")
except Exception as e: print(f"Error configuring Google Gemini: {e}. LLM disabled.")

# E. TTS Setup (Same as before)
print("Initializing TTS engine...")
tts_engine = None
try:
    tts_engine = pyttsx3.init()
    print("TTS engine initialized.")
except Exception as e: print(f"Warning: Could not initialize TTS engine: {e}.")


# --- Step 6: Real-time Processing Threads & Functions ---

# A. Audio Input Stream Callback (Same as before)
audio_stream = None
def audio_callback(indata, frames, time_info, status):
    if status: print("Audio Stream Status:", status, file=sys.stderr) # Print status to stderr
    if not stop_event.is_set():
        try: audio_chunk_q.put(indata.copy(), timeout=0.05) # Shorter timeout
        except queue.Full: pass

# B. Audio Processing Thread (SER Model + Spectrogram - same as before)
def audio_processing_thread_ser():
    global ser_fig, ser_ax, last_detected_voice_emotion_global # Add global voice emotion update
    print("Audio SER/Spectrogram thread started.")
    while not stop_event.is_set():
        try:
            audio_chunk = audio_chunk_q.get(timeout=0.5)
            if audio_chunk.ndim > 1: audio_chunk = audio_chunk.flatten()

            voice_emotion, ser_confidence = analyze_voice_emotion_keras(audio_chunk)
            # Update global voice emotion state
            last_detected_voice_emotion_global = voice_emotion # Simple update (no lock needed if only this thread writes)

            if voice_emotion not in ["Chunk too short", "SER Model Error", "SER Predict Error"]:
                 timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                 with voice_emotion_log_lock:
                     voice_emotion_log.append((timestamp, voice_emotion, ser_confidence))

            if ser_fig is not None and ser_ax is not None and spectrogram_output_widget is not None and spectrogram_output_widget.visible:
                 try:
                     S = librosa.feature.melspectrogram(y=audio_chunk.astype(np.float32), sr=AUDIO_SAMPLE_RATE, n_mels=128, fmax=8000)
                     S_dB = librosa.power_to_db(S, ref=np.max)
                     ser_ax.clear()
                     librosa.display.specshow(S_dB, sr=AUDIO_SAMPLE_RATE, x_axis='time', y_axis='mel', ax=ser_ax, fmax=8000)
                     ser_ax.set_title(f"Live Spectrogram (Voice Emotion: {voice_emotion})")
                     with spectrogram_output_widget: # Assumes correct widget context
                         clear_output(wait=True)
                         display(ser_fig)
                 except Exception as plot_e: print(f"Spectrogram plot error: {plot_e}")

            audio_chunk_q.task_done()
        except queue.Empty: continue
        except Exception as thread_e: print(f"Error in SER/Spectrogram Thread: {thread_e}"); time.sleep(0.1)
    print("Audio SER/Spectrogram thread finished.")

# C. Video Capture & Facial Emotion Thread (NEW!)
video_capture = None
video_display_widget = None # Global ref to the Image widget

def video_capture_face_emotion_thread():
    """Captures video, runs face emotion detection, updates global state and widget."""
    global last_detected_face_emotion_global, video_capture, video_display_widget

    print("Video Capture & Face Emotion thread started.")
    try:
        video_capture = cv2.VideoCapture(0) # Use default webcam
        if not video_capture.isOpened():
            print("Error: Could not open webcam.")
            stop_event.set() # Signal other threads to stop if webcam fails
            return
    except Exception as e:
        print(f"Error initializing webcam: {e}")
        stop_event.set()
        return

    print("Webcam opened successfully.")
    frame_count = 0
    last_analysis_result = {} # Store last successful detection result

    while not stop_event.is_set():
        ret, frame = video_capture.read()
        if not ret:
            print("Error: Failed to grab frame from webcam.")
            time.sleep(0.1)
            continue

        frame_count += 1
        display_frame = frame.copy() # Work on a copy

        # --- Run Face Emotion Detection Periodically ---
        if frame_count % VIDEO_DETECTION_INTERVAL == 0:
            current_face_emotion = "N/A"
            confidence = 0.0
            face_region = None
            try:
                # Use DeepFace.analyze
                # Using 'face_detector_backend' instead of 'detector_backend' for consistency if needed elsewhere
                results = DeepFace.analyze(
                    img_path=frame,
                    actions=['emotion'],
                    detector_backend=DEEPFACE_DETECTOR_BACKEND,
                    enforce_detection=False, # Don't error if no face
                    silent=True # Suppress console logs
                )
                # DeepFace returns a list, analyze the first face found
                if results and isinstance(results, list) and len(results) > 0:
                     analysis = results[0]
                     current_face_emotion = analysis.get('dominant_emotion', "N/A")
                     confidence = analysis.get('emotion', {}).get(current_face_emotion, 0.0) # Get confidence for dominant emotion
                     face_region = analysis.get('region')
                     last_analysis_result = {'emotion': current_face_emotion, 'confidence': confidence, 'region': face_region}
                     # Log the detected face emotion
                     timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                     with face_emotion_log_lock:
                           face_emotion_log.append((timestamp, current_face_emotion, confidence))

                else: # No face detected in this frame
                     last_analysis_result = {} # Clear last result if no face now
                     current_face_emotion = "N/A"

                # Update global state (simple write, no lock needed if only this thread writes)
                last_detected_face_emotion_global = current_face_emotion

            except Exception as e:
                # print(f"DeepFace analysis error: {e}") # Debug if needed
                last_detected_face_emotion_global = "FER Error"
                last_analysis_result = {}


        # --- Draw Box and Emotion on Frame (use last successful result) ---
        if 'region' in last_analysis_result and last_analysis_result['region']:
             r = last_analysis_result['region']
             cv2.rectangle(display_frame, (r['x'], r['y']), (r['x']+r['w'], r['y']+r['h']), (0, 255, 0), 1) # Green box, thinner
             emo = last_analysis_result.get('emotion', 'N/A')
             conf = last_analysis_result.get('confidence', 0.0)
             text = f"{emo} ({conf*100:.0f}%)"
             # Simple text positioning above box
             cv2.putText(display_frame, text, (r['x'], r['y']-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)


        # --- Update Image Widget ---
        if video_display_widget is not None:
            try:
                 # Encode frame to JPEG (faster than PNG)
                 _, encoded_image = cv2.imencode('.jpg', display_frame, [cv2.IMWRITE_JPEG_QUALITY, 80]) # Quality 80%
                 # Update widget value with image bytes
                 video_display_widget.value = encoded_image.tobytes()
            except Exception as e:
                 print(f"Error updating video widget: {e}")

        time.sleep(0.03) # Small delay to prevent 100% CPU, adjust as needed

    # --- Cleanup ---
    print("Video Capture thread finished.")
    if video_capture:
        video_capture.release()
    video_capture = None


# D. Speech-to-Text Thread (Using SpeechRecognition - Same as before)
stt_recognizer = sr.Recognizer()
def speech_to_text_thread():
    # (Keep the STT thread logic exactly as defined in the previous working version)
    print("STT thread started.")
    mic = sr.Microphone(sample_rate=AUDIO_SAMPLE_RATE)
    with mic as source:
        print("STT: Adjusting for ambient noise..."); time.sleep(0.5) # Allow time for user to see message
        try: stt_recognizer.adjust_for_ambient_noise(source, duration=1.0); print(f"STT: Ambient threshold {stt_recognizer.energy_threshold:.1f}")
        except Exception as e: print(f"STT: Ambient adjust failed: {e}")

    while not stop_event.is_set():
        if listen_event.wait(timeout=1.0):
            print("\n>>> STT Listening... <<<"); text = "[STT No Trigger]"
            try:
                with mic as source: audio_data = stt_recognizer.listen(source, phrase_time_limit=STT_PHRASE_TIME_LIMIT, timeout=STT_PHRASE_TIME_LIMIT + 1)
                print(">>> STT Processing... <<<"); text = stt_recognizer.recognize_google(audio_data)
            except sr.WaitTimeoutError: text = "[STT Timeout]"
            except sr.UnknownValueError: text = "[STT Audio Not Understood]"
            except sr.RequestError as e: text = f"[STT API Error]"
            except Exception as e: text = f"[STT Sys Error]"
            finally:
                if text.startswith("["): print(text)
                stt_text_q.put(text); listen_event.clear()
    print("STT thread finished.")


# E. Text-to-Speech Thread (Same as before)
def text_to_speech_thread():
    # (Keep the TTS thread logic exactly as defined in the previous working version)
    print("TTS thread started.")
    while not stop_event.is_set():
        try: text = tts_q.get(timeout=0.5)
        except queue.Empty: continue
        try:
            if tts_engine: tts_engine.say(text); tts_engine.runAndWait()
            tts_q.task_done()
        except Exception as e: print(f"TTS Error: {e}"); tts_q.task_done() # Ensure task done on error
    print("TTS thread finished.")


# --- Step 7: Chatbot Logic Function (Accepts voice and face emotion) ---
def get_chatbot_reply_final(user_text, current_voice_emotion, current_facial_emotion):
    """Generates chatbot reply using NLP, context, emotions, and LLM or fallback."""
    global chat_history
    bot_response = "..."

    if user_text.startswith("[STT") and user_text.endswith("]"): # Handle STT errors
        if "Not Understood" in user_text: return random.choice(["Could you say that again?", "Didn't quite get that."])
        elif "Timeout" in user_text: return random.choice(["Didn't hear you.", "Listening timed out."])
        else: return "Speech input error."

    text_sentiment, _ = analyze_text_nlp_hf(user_text)
    # print(f"[NLP Analysis]: Text Sentiment='{text_sentiment}'") # Less verbose logging

    with chat_history_lock: # Add user msg to history
        chat_history.append({'role': 'user', 'parts': [user_text]})
        limited_history = chat_history[-10:] # Context window

    if llm_model:
        try:
            # --- Enhanced Context Prompt ---
            context_prompt = f"""You are an empathetic AI assistant. Respond kindly. Keep responses concise. If the user mentions severe distress or self-harm, gently advise contacting a crisis hotline (like 988) or professional immediately.

            Current user context:
            - Live voice emotion detected: {current_voice_emotion if current_voice_emotion != 'N/A' else 'Unknown'}
            - Live facial emotion detected: {current_facial_emotion if current_facial_emotion != 'N/A' else 'Unknown'}
            - User text sentiment: {text_sentiment}

            Conversation History:""" # History appended by API call structure

            gemini_content = [{'role': 'user', 'parts': [context_prompt]},
                              {'role': 'model', 'parts': ["Okay, I understand the context. How should I respond empathetically to the user's last message, considering their voice and facial cues?"]}]
            gemini_content.extend(limited_history)

            response = llm_model.generate_content(gemini_content, stream=False, safety_settings={'HARASSMENT':'block_none','HATE_SPEECH': 'block_none','SEXUALLY_EXPLICIT':'block_none','DANGEROUS_CONTENT':'block_none'})

            if response.parts: bot_response = response.text
            else: print(f"Warning: LLM blocked. Feedback: {response.prompt_feedback}"); bot_response = "I can't respond to that specifically. Can we talk about something else?"
        except Exception as e: print(f"LLM Error: {e}"); bot_response = random.choice(["My AI brain fizzled. Try again?", "Connection error."])
    else: # LLM Disabled Fallback
        combined_emotion = f"V: {current_voice_emotion}, F: {current_facial_emotion}"
        if "hello" in user_text.lower(): bot_response = f"Hi there! ({combined_emotion})"
        elif "sad" in user_text.lower(): bot_response = f"Hearing sadness. ({combined_emotion}). I'm listening."
        else: bot_response = f"Okay. ({combined_emotion}). Tell me more."

    with chat_history_lock: # Add bot response to history
        chat_history.append({'role': 'model', 'parts': [bot_response]})
        if len(chat_history) > 40: chat_history = chat_history[-10:]

    if "goodbye quit now" in user_text.lower():
        bot_response = "Okay, ending session. Take care!"
        # Stop sequence initiated later by main loop checking this response
        # But we put it to TTS queue now if needed.
        if tts_engine: tts_q.put(bot_response); tts_q.join() # Wait for final TTS
        stop_event.set() # Signal stop NOW
        print("Exit command recognized.")

    return bot_response


# --- Step 8: Chat Interface Logic ---

def show_chat_mode_selection(username):
    # (Keep chat mode selection logic exactly as before)
    chat_mode = widgets.Dropdown(options=["Text Chat", "Voice Chat", "Video Chat"], description="Chat Mode:")
    start_chat_button = widgets.Button(description="Start Chat")
    chat_mode_output = widgets.Output()

    def on_start_chat(b):
        with chat_mode_output: clear_output(wait=True)
        mode = chat_mode.value
        print(f"Starting {mode} session...")

        # Reset states
        global chat_history, voice_emotion_log, face_emotion_log, stop_event, ser_fig, ser_ax, spectrogram_output_widget, video_display_widget
        stop_event.clear()
        chat_history, voice_emotion_log, face_emotion_log = [], [], []
        ser_fig, ser_ax = None, None
        spectrogram_output_widget, video_display_widget = None, None

        chat_mode.disabled = True
        start_chat_button.disabled = True

        if mode == "Text Chat": text_chat_session(username)
        elif mode == "Voice Chat": voice_chat_session(username)
        elif mode == "Video Chat": video_chat_session(username) # Call the new video session func

    start_chat_button.on_click(on_start_chat)
    with main_area:
        clear_output(wait=True); print("\nPlease select your preferred chat mode:")
        display(widgets.VBox([chat_mode, start_chat_button, chat_mode_output]))


# A. Text Chat Session (Keep as before, ensure it calls end_chat_procedure)
def text_chat_session(username):
    # (Keep the Text Chat session logic exactly as before)
    global chat_history
    text_input = widgets.Textarea(placeholder="Type...", description="You:")
    send_button = widgets.Button(description="Send")
    chat_log_output = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'scroll'})
    end_chat_button = widgets.Button(description="End & Report")

    with chat_log_output:
         clear_output(); initial_msg = "Bot: Text chat active. How can I help?"
         print(initial_msg); chat_history.append({'role': 'model', 'parts': [initial_msg]})
         if tts_engine: tts_q.put(initial_msg) # Optional TTS

    def on_send_clicked(b):
        user_msg = text_input.value; text_input.value = ""
        if not user_msg: return
        with chat_log_output: print(f"\nYou: {user_msg}")
        # Pass N/A for emotions in text mode
        bot_reply = get_chatbot_reply_final(user_msg, current_voice_emotion="N/A", current_facial_emotion="N/A")
        with chat_log_output: print(f"Bot: {bot_reply}")
        if tts_engine: tts_q.put(bot_reply)
        if "ending session" in bot_reply.lower(): end_chat_procedure(username)

    def on_end_chat_clicked(b): end_chat_procedure(username)

    send_button.on_click(on_send_clicked); end_chat_button.on_click(on_end_chat_clicked)
    text_chat_box = widgets.VBox([chat_log_output, text_input, send_button, end_chat_button])
    with main_area: clear_output(wait=True); print("--- Text Chat Session ---"); display(text_chat_box)


# B. Voice Chat Session (Keep as before, using SER/Spectrogram)
def voice_chat_session(username):
    # (Keep the Voice Chat session logic exactly as before)
    # It initializes and uses spectrogram_output_widget globally
    global chat_history, voice_emotion_log, audio_stream, stop_event, ser_fig, ser_ax, spectrogram_output_widget

    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc', 'margin_bottom': '10px'})
    chat_log_output_vc = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'scroll'})
    listen_button = widgets.Button(description="🎤 Speak", button_style='info', icon='microphone')
    end_chat_button_vc = widgets.Button(description="End & Report", button_style='danger')
    status_label = widgets.Label(value="Status: Initializing...")

    ser_fig, ser_ax = plt.subplots(figsize=(7, 4)); plt.close(ser_fig) # Setup plot fig
    ser_ax.set_title("Live Spectrogram - Waiting...")

    voice_chat_box = widgets.VBox([status_label, spectrogram_output_widget, chat_log_output_vc, widgets.HBox([listen_button, end_chat_button_vc])])
    with main_area:
        clear_output(wait=True); print("--- Voice Chat Session ---")
        with spectrogram_output_widget: display(ser_fig) # Show initial empty plot
        display(voice_chat_box)

    status_label.value = "Status: Starting threads..."
    threads = []
    stt_thread = threading.Thread(target=speech_to_text_thread); threads.append(stt_thread); stt_thread.start()
    ser_thread = threading.Thread(target=audio_processing_thread_ser); threads.append(ser_thread); ser_thread.start()
    if tts_engine: tts_thread = threading.Thread(target=text_to_speech_thread); threads.append(tts_thread); tts_thread.start()

    try:
        status_label.value = "Status: Starting audio..."
        audio_stream = sd.InputStream(samplerate=AUDIO_SAMPLE_RATE, channels=AUDIO_CHANNELS, blocksize=AUDIO_CHUNK_SAMPLES, callback=audio_callback, dtype='float32')
        audio_stream.start()
        status_label.value = "Status: Ready. Speak or press Speak button."
        with chat_log_output_vc: initial_msg="Bot: Voice chat active."; print(initial_msg)
        if tts_engine: tts_q.put(initial_msg)
        chat_history.append({'role': 'model', 'parts': [initial_msg]})
    except Exception as e: status_label.value = f"Status: ERROR starting audio: {e}"; print(f"Audio Error: {e}"); stop_event.set(); return

    def on_listen_button_clicked(b): listen_event.set(); status_label.value = "Status: Listening actively..."
    def on_end_vc_clicked(b): status_label.value = "Status: Ending..."; stop_event.set(); listen_button.disabled = True; end_chat_button_vc.disabled = True

    listen_button.on_click(on_listen_button_clicked)
    end_chat_button_vc.on_click(on_end_vc_clicked)

    # Main interaction loop is handled outside by run_prototype checking stt_text_q


# C. Integrated Video Chat Session (NEW IMPLEMENTATION!)
def video_chat_session(username):
    """Starts video, audio, SER, Spectrogram, STT, TTS, and interaction."""
    global chat_history, voice_emotion_log, face_emotion_log, audio_stream, video_capture, stop_event
    global ser_fig, ser_ax, spectrogram_output_widget, video_display_widget # Access global widget refs

    print("Initializing Video Chat Session...")

    # --- UI Setup ---
    status_label_vc = widgets.Label(value="Status: Initializing...")

    # Video display widget
    video_display_widget = widgets.Image(format='jpeg', width=480) # Adjust width as needed
    video_box = widgets.VBox([widgets.Label("Live Video Feed:"), video_display_widget], layout={'border': '1px solid grey'})

    # Spectrogram display widget
    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc'})
    spectrogram_box = widgets.VBox([widgets.Label("Live Audio Spectrogram:"), spectrogram_output_widget], layout={'margin_top': '10px'})
    # Initialize the plot figure and axes for spectrogram
    ser_fig, ser_ax = plt.subplots(figsize=(6, 3)) # Smaller spectrogram for video mode?
    ser_ax.set_title("Live Spectrogram - Waiting...")
    plt.close(ser_fig) # Close the initial display

    # Conversation log
    chat_log_output_vc = widgets.Output(layout={'border': '1px solid black', 'height': '250px', 'overflow_y': 'scroll', 'margin_top': '10px'})

    # Controls
    listen_button_vc = widgets.Button(description="🎤 Speak", button_style='info', icon='microphone')
    end_chat_button_vc = widgets.Button(description="End Video Chat & Report", button_style='danger')
    controls_box = widgets.HBox([listen_button_vc, end_chat_button_vc], layout={'margin_top': '10px'})

    # Arrange UI (e.g., Video and Spectrogram side-by-side, chat below)
    top_row = widgets.HBox([video_box, spectrogram_box])
    video_chat_main_box = widgets.VBox([status_label_vc, top_row, chat_log_output_vc, controls_box])

    # Display the UI
    with main_area:
        clear_output(wait=True)
        print("--- Video Chat Session ---")
        # Display placeholder plot initially in its widget
        with spectrogram_output_widget: display(ser_fig)
        display(video_chat_main_box)


    # --- Start Threads & Streams ---
    status_label_vc.value = "Status: Starting threads..."
    print("Starting threads for video session...")
    stop_event.clear()
    chat_history, voice_emotion_log, face_emotion_log = [], [], [] # Reset logs

    threads = []
    # Video Thread
    vid_thread = threading.Thread(target=video_capture_face_emotion_thread); threads.append(vid_thread); vid_thread.start()
    # Audio SER/Spectrogram Thread
    ser_thread = threading.Thread(target=audio_processing_thread_ser); threads.append(ser_thread); ser_thread.start()
    # STT Thread
    stt_thread = threading.Thread(target=speech_to_text_thread); threads.append(stt_thread); stt_thread.start()
    # TTS Thread
    if tts_engine: tts_thread = threading.Thread(target=text_to_speech_thread); threads.append(tts_thread); tts_thread.start()


    # Start Audio Stream
    try:
        status_label_vc.value = "Status: Starting audio..."
        audio_stream = sd.InputStream(samplerate=AUDIO_SAMPLE_RATE, channels=AUDIO_CHANNELS, blocksize=AUDIO_CHUNK_SAMPLES, callback=audio_callback, dtype='float32')
        audio_stream.start()
        status_label_vc.value = "Status: Ready. Speak or press Speak button."
        print("Audio stream started for video session.")

        with chat_log_output_vc: initial_msg="Bot: Video & audio chat active."; print(initial_msg)
        if tts_engine: tts_q.put(initial_msg)
        chat_history.append({'role': 'model', 'parts': [initial_msg]})

    except Exception as e:
        status_label_vc.value = f"Status: ERROR starting audio stream: {e}"
        print(f"Audio Stream Error: {e}"); stop_event.set(); return # Stop if audio fails


    # --- Button Actions ---
    def on_listen_vc_clicked(b): listen_event.set(); status_label_vc.value = "Status: Listening actively..."
    def on_end_vchat_clicked(b):
         status_label_vc.value = "Status: Ending...";
         stop_event.set(); # Signal threads & stream to stop
         listen_button_vc.disabled=True; end_chat_button_vc.disabled=True

    listen_button_vc.on_click(on_listen_vc_clicked)
    end_chat_button_vc.on_click(on_end_vchat_clicked)

    # Main interaction loop is handled outside by run_prototype checking stt_text_q


# --- Step 9: Session Report Generation (Updated to include Face Emotion) ---
def generate_report(username):
    # ... (keep report header, intake, initial assessment sections same as before) ...
    data = user_db[username]["data"]
    report_lines = []
    report_lines.append("="*40); report_lines.append("  Multi-Modal Chatbot Session Report"); report_lines.append("="*40)
    report_lines.append(f"User: {username}"); report_lines.append(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    report_lines.append("--- User Intake Summary ---")
    if data.get("intake"): [report_lines.append(f"  {k.capitalize()}: {v}") for k, v in data["intake"].items()]
    else: report_lines.append("  (No intake data)")
    report_lines.append("\n--- Initial Assessment (Experimental) ---")
    if data.get("issues"): [report_lines.append(f"  Potential for {k}: {int(v * 100)}% likelihood") for k, v in data["issues"].items()]
    else: report_lines.append("  (No assessment data)")

    # --- Add Voice Emotion Log ---
    report_lines.append("\n--- Voice Emotion Log ---")
    if voice_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence")
         report_lines.append("  ------------|-----------------|-----------")
         with voice_emotion_log_lock: # Access log safely
             log_limit = 30; display_log = voice_emotion_log[-log_limit:]
             if len(voice_emotion_log) > log_limit: report_lines.append("  ...(last 30 entries)...")
             for ts, emo, conf in display_log: report_lines.append(f"  {ts} | {emo:<15} | {conf:.2f}")
    else: report_lines.append("  (No voice emotions logged for this session)")

    # --- Add Face Emotion Log (NEW) ---
    report_lines.append("\n--- Face Emotion Log ---")
    if face_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence")
         report_lines.append("  ------------|-----------------|-----------")
         with face_emotion_log_lock: # Access log safely
             log_limit = 30; display_log = face_emotion_log[-log_limit:]
             if len(face_emotion_log) > log_limit: report_lines.append("  ...(last 30 entries)...")
             for ts, emo, conf in display_log: report_lines.append(f"  {ts} | {emo:<15} | {conf:.2f}")
    else: report_lines.append("  (No face emotions logged for this session)")
    # ------------------------------------

    report_lines.append("\n--- Conversation Transcript ---")
    if chat_history: [report_lines.append(f"  {t.get('role','?').capitalize()}: {t.get('parts',[''])[0]}") for t in chat_history]
    else: report_lines.append("  (No conversation recorded)")

    report_lines.append("\n"+"="*40)
    report_content = "\n".join(report_lines)
    filename = f"{username}_session_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    try:
        with open(filename, "w", encoding='utf-8') as f: f.write(report_content)
        print(f"\nSession report generated: '{filename}'")
    except Exception as e: print(f"\nError writing report: {e}")

    # --- Professional Help Suggestion (Same as before) ---
    issues = data.get("issues", {})
    if any(prob > 0.65 for prob in issues.values()):
         print("\nRecommendation: Consider reaching out to a mental health professional.")
         print("Resources: 988 Lifeline (US/Canada), Befrienders Worldwide, Crisis Text Line.")
    else: print("\nRecommendation: Continue self-care. Consult a professional if needed.")

# (Keep end_chat_procedure exactly as before - it stops threads/stream and calls generate_report)
def end_chat_procedure(username):
    print("\nEnding chat session and generating report...")
    stop_event.set()
    global audio_stream, video_capture # Ensure we clear global captures
    if audio_stream is not None and audio_stream.active:
        try: print("Stopping audio stream..."); audio_stream.stop(); audio_stream.close(); print("Audio stopped.")
        except Exception as e: print(f"Error stopping audio: {e}")
        finally: audio_stream = None
    if video_capture is not None and video_capture.isOpened(): # Add video cleanup
        try: print("Releasing video capture..."); video_capture.release(); print("Video released.")
        except Exception as e: print(f"Error releasing video: {e}")
        finally: video_capture = None
    cv2.destroyAllWindows() # Close any OpenCV windows just in case
    for i in range(3): cv2.waitKey(1)
    print("Waiting briefly for threads...")
    time.sleep(1.5) # Slightly longer wait?
    with main_area:
        clear_output(wait=True); generate_report(username)
        print("\nSession ended. Run the first cell again to log in.")


# --- Step 10: Main Application Execution Logic (Updated Interaction Loop) ---
def run_prototype():
    clear_output(); print("="*50); print(" Multi-Modal Mental Health Chatbot Prototype"); print("="*50)
    display(login_box); display(main_area)

    try:
         print("\nPrototype running. Login to start.")
         while not stop_event.is_set():
            try:
                 user_text = stt_text_q.get(timeout=0.5)

                 # Find appropriate chat log widget (Needs better state mgmt)
                 current_chat_log_widget = main_area # Default to main

                 # --- Process User Speech ---
                 if user_text:
                     print(f"\n[STT Received]: {user_text}") # Debug
                     with current_chat_log_widget: # Display in the main area for now
                          print(f"You (voice): {user_text}")

                     # Fetch latest emotions (use globals directly - read is often okay without lock if writes are atomic/infrequent)
                     # Use defaults if not updated yet
                     current_voice_emotion = last_detected_voice_emotion_global
                     current_facial_emotion = last_detected_face_emotion_global

                     # Get bot reply using both emotions
                     bot_reply = get_chatbot_reply_final(user_text, current_voice_emotion, current_facial_emotion)

                     with current_chat_log_widget: # Display bot reply
                         print(f"Bot: {bot_reply}")
                     if tts_engine: tts_q.put(bot_reply)

                     # Check if the bot response signals an end
                     if "ending session" in bot_reply.lower():
                           print("Bot initiated session end.")
                           # Ensure TTS finishes before full stop if possible
                           if tts_engine: tts_q.join()
                           stop_event.set() # Signal threads after response processed

                 stt_text_q.task_done() # Mark STT task as done

            except queue.Empty: time.sleep(0.1); continue # No text, just wait
            except Exception as main_loop_e: print(f"!! Error in main loop: {main_loop_e} !!"); time.sleep(1)

    except KeyboardInterrupt: print("\nKeyboard Interrupt! Stopping..."); stop_event.set()
    finally:
         print("Main loop exited.")
         if not stop_event.is_set(): # Ensure cleanup if loop broke unexpectedly
            stop_event.set()
         # Run cleanup directly if needed (end_chat_procedure might be better called explicitly via button)
         # Cleanup shared resources that might linger
         global audio_stream, video_capture
         if audio_stream is not None: audio_stream.close(); audio_stream = None
         if video_capture is not None: video_capture.release(); video_capture = None
         cv2.destroyAllWindows()
         print("Prototype Shutdown Complete.")


# --- Start the Prototype ---
run_prototype()

 Multi-Modal Mental Health Chatbot Prototype


Output()


Prototype running. Login to start.


In [ ]:
# ========================
# FINAL INTEGRATED PROTOTYPE CODE (Enhanced with Live Voice/Face SER/Spectrogram in VIDEO mode)
# ========================
# Run this cell first for installs!
# %pip install ipywidgets matplotlib numpy librosa tensorflow scikit-learn transformers sounddevice SpeechRecognition pyttsx3 google-generativeai pydub deepface opencv-python # Added deepface, opencv

# --- Step 1: Imports & Setup ---
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
import matplotlib.pyplot as plt
import numpy as np
import random
import datetime
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from transformers import pipeline
import sounddevice as sd
import speech_recognition as sr
import pyttsx3
import google.generativeai as genai
import threading
import queue
import time
import os
from pydub import AudioSegment
import io
import cv2 # Added for video capture
from deepface import DeepFace # Added for facial emotion recognition

# For reproducibility and silencing TF/DeepFace warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
# logging.getLogger("tensorflow").setLevel(logging.ERROR) # Alternate way
# logging.getLogger("deepface").setLevel(logging.WARN)    # Reduce deepface spam
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries loaded.")

# --- Global Config & Resources ---
AUDIO_SAMPLE_RATE = 16000
AUDIO_CHANNELS = 1
AUDIO_CHUNK_DURATION = 0.5
AUDIO_CHUNK_SAMPLES = int(AUDIO_SAMPLE_RATE * AUDIO_CHUNK_DURATION)
STT_PHRASE_TIME_LIMIT = 7
# Video/DeepFace settings
VIDEO_DETECTION_INTERVAL = 5 # Frames between face detection runs
DEEPFACE_DETECTOR_BACKEND = 'opencv' # Fast but less accurate; try 'mtcnn', 'ssd' etc. if needed

# Queues & Events
stt_text_q = queue.Queue()
tts_q = queue.Queue()
audio_chunk_q = queue.Queue(maxsize=20)
video_frame_q = queue.Queue(maxsize=5) # Queue for frames to display widget
stop_event = threading.Event()
listen_event = threading.Event()

# Shared State (with locks)
chat_history_lock = threading.Lock()
chat_history = []
voice_emotion_log_lock = threading.Lock()
voice_emotion_log = [] # List of (timestamp, detected_emotion, confidence)
face_emotion_log_lock = threading.Lock()
face_emotion_log = [] # List of (timestamp, detected_emotion, confidence)

# Global emotion holders (updated by threads)
last_detected_voice_emotion_global = "N/A"
last_detected_face_emotion_global = "N/A"

# Matplotlib Plot variables (for spectrogram)
ser_fig = None
ser_ax = None
spectrogram_output_widget = None # Will be set depending on chat mode

print("Shared resources initialized.")

# --- (Steps 2, 3, 4: Auth, Intake, Dummy Classification - UNCHANGED from previous) ---
# Keep the code for user_db, create_account, login, show_intake_form,
# train_mental_health_classifier, simulate_classification here...
# --- BEGIN Steps 2,3,4 ---
user_db = {} # Simulated user database

def create_account(username, password):
    if username in user_db: return False, "Username exists."
    user_db[username] = {"password": password, "data": {}}
    return True, "Account created."

def login(username, password):
    if username in user_db and user_db[username]["password"] == password: return True, "Login successful."
    return False, "Invalid credentials."

# Widgets for login
login_username = widgets.Text(description="Username:")
login_password = widgets.Password(description="Password:")
login_button = widgets.Button(description="Login")
signup_button = widgets.Button(description="Sign Up")
login_output = widgets.Output()
main_area = widgets.Output() # Area to display next steps after login

current_username = None # Store logged in user

def on_login_clicked(b):
    global current_username
    with login_output:
        clear_output(wait=True)
        user = login_username.value; pwd = login_password.value
        success, msg = login(user, pwd)
        print(msg)
        if success:
            current_username = user
            login_box.layout.display = 'none' # Hide login
            with main_area:
                clear_output(wait=True)
                show_intake_form(user)

def on_signup_clicked(b):
    with login_output:
        clear_output(wait=True)
        user = login_username.value; pwd = login_password.value
        success, msg = create_account(user, pwd)
        print(msg)

login_button.on_click(on_login_clicked)
signup_button.on_click(on_signup_clicked)
login_box = widgets.VBox([login_username, login_password, widgets.HBox([login_button, signup_button]), login_output])


# Step 3: Demographic & Intake (From main_prototype - UNCHANGED)
def show_intake_form(username):
    intake_form_box_widgets = []
    age = widgets.IntText(description="Age:", value=30)
    gender = widgets.Dropdown(options=["Male", "Female", "Non-binary", "Other", "Prefer not to say"], description="Gender:")
    ethnicity = widgets.Text(description="Ethnicity (Optional):")
    socioeconomic = widgets.Dropdown(options=["Low", "Medium", "High", "Prefer not to say"], description="Socioeconomic Status (Optional):")
    sleep = widgets.IntSlider(value=7, min=0, max=14, description="Avg Sleep (hrs):")
    appetite = widgets.Dropdown(options=["Normal", "Decreased", "Increased", "Fluctuates"], description="Appetite Change:")
    energy = widgets.IntSlider(value=5, min=0, max=10, description="Energy Level (0-10):")
    social = widgets.Dropdown(options=["Very Active", "Moderately Active", "Somewhat Withdrawn", "Very Withdrawn"], description="Social Activity:")
    intake_form_box_widgets.extend([age, gender, ethnicity, socioeconomic, sleep, appetite, energy, social])

    submit_button = widgets.Button(description="Submit Intake")
    intake_output = widgets.Output()

    def on_submit_clicked(b):
        with main_area: # Display output in the main area
             clear_output(wait=True)
             user_db[username]["data"]["intake"] = {
                "age": age.value, "gender": gender.value, "ethnicity": ethnicity.value,
                "socioeconomic": socioeconomic.value, "sleep": sleep.value,
                "appetite": appetite.value, "energy": energy.value, "social": social.value
             }
             print("Intake data submitted.")
             simulate_classification(username) # Proceed to classification

    submit_button.on_click(on_submit_clicked)
    intake_box = widgets.VBox([*intake_form_box_widgets, submit_button, intake_output])
    with main_area:
         clear_output(wait=True)
         print(f"Welcome {username}! Please provide some optional details.")
         display(intake_box)

# Step 4: Model 4 – Mental Health Classification (Dummy - From main_prototype - UNCHANGED)
def train_mental_health_classifier():
    X_train = np.array([[25, 7, 6, 3],[40, 5, 4, 1],[30, 8, 7, 4],[22, 6, 5, 2], [55, 4, 3, 1], [35, 9, 8, 3]])
    y_train = np.array([0, 1, 0, 1, 1, 0]) # 0=Low risk, 1=High risk
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_train, y_train)
    return clf
mental_health_classifier = train_mental_health_classifier()

def simulate_classification(username):
    intake = user_db[username]["data"].get("intake", {})
    if not intake: print("No intake data."); return
    # Simplified feature mapping for the dummy model
    features = np.array([[
        intake.get("age", 30), intake.get("sleep", 7), intake.get("energy", 5),
        {"Very Active": 3, "Moderately Active": 2, "Somewhat Withdrawn": 1, "Very Withdrawn": 0}.get(intake.get("social"), 1)
    ]])
    risk = mental_health_classifier.predict(features)[0]
    prob = mental_health_classifier.predict_proba(features)[0][risk]
    issues = {"Depression": 0.0, "Anxiety": 0.0, "Stress & Burnout": 0.0}
    if risk == 1:
        issues["Depression"] = round(min(1.0, prob + random.uniform(0.1, 0.4)), 2)
        issues["Anxiety"] = round(min(1.0, prob + random.uniform(0.2, 0.5)), 2)
        issues["Stress & Burnout"] = round(min(1.0, prob + random.uniform(0.3, 0.6)), 2)
    else:
        issues["Depression"] = round(max(0.0, prob - random.uniform(0.1, 0.3)), 2)
        issues["Anxiety"] = round(max(0.0, prob - random.uniform(0.2, 0.4)), 2)
        issues["Stress & Burnout"] = round(max(0.0, prob - random.uniform(0.1, 0.2)), 2)

    user_db[username]["data"]["issues"] = issues
    print("\nBased on intake data (experimental prediction):")
    for issue, p in issues.items(): print(f" - {issue}: {int(p * 100)}% likelihood")
    show_chat_mode_selection(username) # Proceed to chat mode selection
# --- END Steps 2,3,4 ---


# --- Step 5: Load Models ---
# A. Face Emotion Model (We'll use DeepFace analyze directly, no separate Keras model needed unless customizing)
FACE_EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"] # Standard DeepFace output

# B. Voice Emotion Model (Load Keras LSTM)
SER_MODEL_PATH = 'speech_emotion_model.h5'
ser_model = None; ser_encoder = None
SER_EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad'] # Match TESS labels used for training
try:
    ser_encoder = OneHotEncoder(); ser_encoder.fit(np.array(SER_EMOTIONS).reshape(-1, 1))
    ser_model = load_model(SER_MODEL_PATH)
    print(f"Loaded SER model from {SER_MODEL_PATH}")
    # ser_model.summary()
except Exception as e:
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print(f"ERROR loading SER model from {SER_MODEL_PATH}: {e}")
    print(f"Voice Emotion Analysis will NOT work. Make sure the model file exists.")
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    ser_model = None
# (Keep analyze_voice_emotion_keras function as defined before)
def analyze_voice_emotion_keras(audio_chunk):
    if ser_model is None or ser_encoder is None: return "SER Model Error", 0.0
    emotion, confidence = "unknown", 0.0
    try:
        if audio_chunk.dtype != np.float32: audio_chunk = audio_chunk.astype(np.float32) / 32767.0 # Normalize INT16 etc.
        if len(audio_chunk) < 2048: return "Chunk too short", 0.0
        mfcc = np.mean(librosa.feature.mfcc(y=audio_chunk, sr=AUDIO_SAMPLE_RATE, n_mfcc=40).T, axis=0)
        mfcc_reshaped = np.expand_dims(np.expand_dims(mfcc, axis=0), axis=-1)
        prediction = ser_model.predict(mfcc_reshaped, verbose=0)
        emotion_index = np.argmax(prediction[0])
        confidence = prediction[0][emotion_index]
        emotion = ser_encoder.inverse_transform(np.array(emotion_index).reshape(1, -1))[0][0]
    except Exception as e: print(f"SER Prediction Error: {e}"); emotion = "SER Predict Error"
    return emotion, confidence


# C. NLP Text Analysis Pipeline (Same as before)
print("Loading NLP pipeline...")
nlp_text_analyzer = None
try:
    nlp_text_analyzer = pipeline("sentiment-analysis")
    print("NLP pipeline loaded.")
except Exception as e: print(f"Warning: Failed to load NLP pipeline: {e}.")
# (Keep analyze_text_nlp_hf function as defined before)
def analyze_text_nlp_hf(text):
     if nlp_text_analyzer:
         try: result = nlp_text_analyzer(text)[0]; return result["label"], result["score"]
         except Exception as e: print(f"NLP analysis failed: {e}"); return "NLP Error", 0.0
     else: # Basic fallback
         if "sad" in text.lower(): return "NEGATIVE", 0.7
         if "happy" in text.lower(): return "POSITIVE", 0.7
         return "NEUTRAL", 0.5

# D. LLM Setup (Gemini - same as before)
print("Configuring LLM...")
llm_model = None
try: # Ensure GOOGLE_API_KEY is set as environment variable BEFORE launching Jupyter
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
    if GOOGLE_API_KEY:
        genai.configure(api_key=GOOGLE_API_KEY)
        llm_model = genai.GenerativeModel('gemini-pro') # Or 'gemini-1.5-flash'
        print("Google Gemini configured.")
    else: print("Warning: GOOGLE_API_KEY environment variable not found. LLM disabled.")
except Exception as e: print(f"Error configuring Google Gemini: {e}. LLM disabled.")

# E. TTS Setup (Same as before)
print("Initializing TTS engine...")
tts_engine = None
try:
    tts_engine = pyttsx3.init()
    print("TTS engine initialized.")
except Exception as e: print(f"Warning: Could not initialize TTS engine: {e}.")


# --- Step 6: Real-time Processing Threads & Functions ---

# A. Audio Input Stream Callback (Same as before)
audio_stream = None
def audio_callback(indata, frames, time_info, status):
    if status: print("Audio Stream Status:", status, file=sys.stderr) # Print status to stderr
    if not stop_event.is_set():
        try: audio_chunk_q.put(indata.copy(), timeout=0.05) # Shorter timeout
        except queue.Full: pass

# B. Audio Processing Thread (SER Model + Spectrogram - same as before)
def audio_processing_thread_ser():
    global ser_fig, ser_ax, last_detected_voice_emotion_global # Add global voice emotion update
    print("Audio SER/Spectrogram thread started.")
    while not stop_event.is_set():
        try:
            audio_chunk = audio_chunk_q.get(timeout=0.5)
            if audio_chunk.ndim > 1: audio_chunk = audio_chunk.flatten()

            voice_emotion, ser_confidence = analyze_voice_emotion_keras(audio_chunk)
            # Update global voice emotion state
            last_detected_voice_emotion_global = voice_emotion # Simple update (no lock needed if only this thread writes)

            if voice_emotion not in ["Chunk too short", "SER Model Error", "SER Predict Error"]:
                 timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                 with voice_emotion_log_lock:
                     voice_emotion_log.append((timestamp, voice_emotion, ser_confidence))

            if ser_fig is not None and ser_ax is not None and spectrogram_output_widget is not None and spectrogram_output_widget.visible:
                 try:
                     S = librosa.feature.melspectrogram(y=audio_chunk.astype(np.float32), sr=AUDIO_SAMPLE_RATE, n_mels=128, fmax=8000)
                     S_dB = librosa.power_to_db(S, ref=np.max)
                     ser_ax.clear()
                     librosa.display.specshow(S_dB, sr=AUDIO_SAMPLE_RATE, x_axis='time', y_axis='mel', ax=ser_ax, fmax=8000)
                     ser_ax.set_title(f"Live Spectrogram (Voice Emotion: {voice_emotion})")
                     with spectrogram_output_widget: # Assumes correct widget context
                         clear_output(wait=True)
                         display(ser_fig)
                 except Exception as plot_e: print(f"Spectrogram plot error: {plot_e}")

            audio_chunk_q.task_done()
        except queue.Empty: continue
        except Exception as thread_e: print(f"Error in SER/Spectrogram Thread: {thread_e}"); time.sleep(0.1)
    print("Audio SER/Spectrogram thread finished.")

# C. Video Capture & Facial Emotion Thread (NEW!)
video_capture = None
video_display_widget = None # Global ref to the Image widget

def video_capture_face_emotion_thread():
    """Captures video, runs face emotion detection, updates global state and widget."""
    global last_detected_face_emotion_global, video_capture, video_display_widget

    print("Video Capture & Face Emotion thread started.")
    try:
        video_capture = cv2.VideoCapture(0) # Use default webcam
        if not video_capture.isOpened():
            print("Error: Could not open webcam.")
            stop_event.set() # Signal other threads to stop if webcam fails
            return
    except Exception as e:
        print(f"Error initializing webcam: {e}")
        stop_event.set()
        return

    print("Webcam opened successfully.")
    frame_count = 0
    last_analysis_result = {} # Store last successful detection result

    while not stop_event.is_set():
        ret, frame = video_capture.read()
        if not ret:
            print("Error: Failed to grab frame from webcam.")
            time.sleep(0.1)
            continue

        frame_count += 1
        display_frame = frame.copy() # Work on a copy

        # --- Run Face Emotion Detection Periodically ---
        if frame_count % VIDEO_DETECTION_INTERVAL == 0:
            current_face_emotion = "N/A"
            confidence = 0.0
            face_region = None
            try:
                # Use DeepFace.analyze
                # Using 'face_detector_backend' instead of 'detector_backend' for consistency if needed elsewhere
                results = DeepFace.analyze(
                    img_path=frame,
                    actions=['emotion'],
                    detector_backend=DEEPFACE_DETECTOR_BACKEND,
                    enforce_detection=False, # Don't error if no face
                    silent=True # Suppress console logs
                )
                # DeepFace returns a list, analyze the first face found
                if results and isinstance(results, list) and len(results) > 0:
                     analysis = results[0]
                     current_face_emotion = analysis.get('dominant_emotion', "N/A")
                     confidence = analysis.get('emotion', {}).get(current_face_emotion, 0.0) # Get confidence for dominant emotion
                     face_region = analysis.get('region')
                     last_analysis_result = {'emotion': current_face_emotion, 'confidence': confidence, 'region': face_region}
                     # Log the detected face emotion
                     timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                     with face_emotion_log_lock:
                           face_emotion_log.append((timestamp, current_face_emotion, confidence))

                else: # No face detected in this frame
                     last_analysis_result = {} # Clear last result if no face now
                     current_face_emotion = "N/A"

                # Update global state (simple write, no lock needed if only this thread writes)
                last_detected_face_emotion_global = current_face_emotion

            except Exception as e:
                # print(f"DeepFace analysis error: {e}") # Debug if needed
                last_detected_face_emotion_global = "FER Error"
                last_analysis_result = {}


        # --- Draw Box and Emotion on Frame (use last successful result) ---
        if 'region' in last_analysis_result and last_analysis_result['region']:
             r = last_analysis_result['region']
             cv2.rectangle(display_frame, (r['x'], r['y']), (r['x']+r['w'], r['y']+r['h']), (0, 255, 0), 1) # Green box, thinner
             emo = last_analysis_result.get('emotion', 'N/A')
             conf = last_analysis_result.get('confidence', 0.0)
             text = f"{emo} ({conf*100:.0f}%)"
             # Simple text positioning above box
             cv2.putText(display_frame, text, (r['x'], r['y']-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)


        # --- Update Image Widget ---
        if video_display_widget is not None:
            try:
                 # Encode frame to JPEG (faster than PNG)
                 _, encoded_image = cv2.imencode('.jpg', display_frame, [cv2.IMWRITE_JPEG_QUALITY, 80]) # Quality 80%
                 # Update widget value with image bytes
                 video_display_widget.value = encoded_image.tobytes()
            except Exception as e:
                 print(f"Error updating video widget: {e}")

        time.sleep(0.03) # Small delay to prevent 100% CPU, adjust as needed

    # --- Cleanup ---
    print("Video Capture thread finished.")
    if video_capture:
        video_capture.release()
    video_capture = None


# D. Speech-to-Text Thread (Using SpeechRecognition - Same as before)
stt_recognizer = sr.Recognizer()
def speech_to_text_thread():
    # (Keep the STT thread logic exactly as defined in the previous working version)
    print("STT thread started.")
    mic = sr.Microphone(sample_rate=AUDIO_SAMPLE_RATE)
    with mic as source:
        print("STT: Adjusting for ambient noise..."); time.sleep(0.5) # Allow time for user to see message
        try: stt_recognizer.adjust_for_ambient_noise(source, duration=1.0); print(f"STT: Ambient threshold {stt_recognizer.energy_threshold:.1f}")
        except Exception as e: print(f"STT: Ambient adjust failed: {e}")

    while not stop_event.is_set():
        if listen_event.wait(timeout=1.0):
            print("\n>>> STT Listening... <<<"); text = "[STT No Trigger]"
            try:
                with mic as source: audio_data = stt_recognizer.listen(source, phrase_time_limit=STT_PHRASE_TIME_LIMIT, timeout=STT_PHRASE_TIME_LIMIT + 1)
                print(">>> STT Processing... <<<"); text = stt_recognizer.recognize_google(audio_data)
            except sr.WaitTimeoutError: text = "[STT Timeout]"
            except sr.UnknownValueError: text = "[STT Audio Not Understood]"
            except sr.RequestError as e: text = f"[STT API Error]"
            except Exception as e: text = f"[STT Sys Error]"
            finally:
                if text.startswith("["): print(text)
                stt_text_q.put(text); listen_event.clear()
    print("STT thread finished.")


# E. Text-to-Speech Thread (Same as before)
def text_to_speech_thread():
    # (Keep the TTS thread logic exactly as defined in the previous working version)
    print("TTS thread started.")
    while not stop_event.is_set():
        try: text = tts_q.get(timeout=0.5)
        except queue.Empty: continue
        try:
            if tts_engine: tts_engine.say(text); tts_engine.runAndWait()
            tts_q.task_done()
        except Exception as e: print(f"TTS Error: {e}"); tts_q.task_done() # Ensure task done on error
    print("TTS thread finished.")


# --- Step 7: Chatbot Logic Function (Accepts voice and face emotion) ---
def get_chatbot_reply_final(user_text, current_voice_emotion, current_facial_emotion):
    """Generates chatbot reply using NLP, context, emotions, and LLM or fallback."""
    global chat_history
    bot_response = "..."

    if user_text.startswith("[STT") and user_text.endswith("]"): # Handle STT errors
        if "Not Understood" in user_text: return random.choice(["Could you say that again?", "Didn't quite get that."])
        elif "Timeout" in user_text: return random.choice(["Didn't hear you.", "Listening timed out."])
        else: return "Speech input error."

    text_sentiment, _ = analyze_text_nlp_hf(user_text)
    # print(f"[NLP Analysis]: Text Sentiment='{text_sentiment}'") # Less verbose logging

    with chat_history_lock: # Add user msg to history
        chat_history.append({'role': 'user', 'parts': [user_text]})
        limited_history = chat_history[-10:] # Context window

    if llm_model:
        try:
            # --- Enhanced Context Prompt ---
            context_prompt = f"""You are an empathetic AI assistant. Respond kindly. Keep responses concise. If the user mentions severe distress or self-harm, gently advise contacting a crisis hotline (like 988) or professional immediately.

            Current user context:
            - Live voice emotion detected: {current_voice_emotion if current_voice_emotion != 'N/A' else 'Unknown'}
            - Live facial emotion detected: {current_facial_emotion if current_facial_emotion != 'N/A' else 'Unknown'}
            - User text sentiment: {text_sentiment}

            Conversation History:""" # History appended by API call structure

            gemini_content = [{'role': 'user', 'parts': [context_prompt]},
                              {'role': 'model', 'parts': ["Okay, I understand the context. How should I respond empathetically to the user's last message, considering their voice and facial cues?"]}]
            gemini_content.extend(limited_history)

            response = llm_model.generate_content(gemini_content, stream=False, safety_settings={'HARASSMENT':'block_none','HATE_SPEECH': 'block_none','SEXUALLY_EXPLICIT':'block_none','DANGEROUS_CONTENT':'block_none'})

            if response.parts: bot_response = response.text
            else: print(f"Warning: LLM blocked. Feedback: {response.prompt_feedback}"); bot_response = "I can't respond to that specifically. Can we talk about something else?"
        except Exception as e: print(f"LLM Error: {e}"); bot_response = random.choice(["My AI brain fizzled. Try again?", "Connection error."])
    else: # LLM Disabled Fallback
        combined_emotion = f"V: {current_voice_emotion}, F: {current_facial_emotion}"
        if "hello" in user_text.lower(): bot_response = f"Hi there! ({combined_emotion})"
        elif "sad" in user_text.lower(): bot_response = f"Hearing sadness. ({combined_emotion}). I'm listening."
        else: bot_response = f"Okay. ({combined_emotion}). Tell me more."

    with chat_history_lock: # Add bot response to history
        chat_history.append({'role': 'model', 'parts': [bot_response]})
        if len(chat_history) > 40: chat_history = chat_history[-10:]

    if "goodbye quit now" in user_text.lower():
        bot_response = "Okay, ending session. Take care!"
        # Stop sequence initiated later by main loop checking this response
        # But we put it to TTS queue now if needed.
        if tts_engine: tts_q.put(bot_response); tts_q.join() # Wait for final TTS
        stop_event.set() # Signal stop NOW
        print("Exit command recognized.")

    return bot_response


# --- Step 8: Chat Interface Logic ---

def show_chat_mode_selection(username):
    # (Keep chat mode selection logic exactly as before)
    chat_mode = widgets.Dropdown(options=["Text Chat", "Voice Chat", "Video Chat"], description="Chat Mode:")
    start_chat_button = widgets.Button(description="Start Chat")
    chat_mode_output = widgets.Output()

    def on_start_chat(b):
        with chat_mode_output: clear_output(wait=True)
        mode = chat_mode.value
        print(f"Starting {mode} session...")

        # Reset states
        global chat_history, voice_emotion_log, face_emotion_log, stop_event, ser_fig, ser_ax, spectrogram_output_widget, video_display_widget
        stop_event.clear()
        chat_history, voice_emotion_log, face_emotion_log = [], [], []
        ser_fig, ser_ax = None, None
        spectrogram_output_widget, video_display_widget = None, None

        chat_mode.disabled = True
        start_chat_button.disabled = True

        if mode == "Text Chat": text_chat_session(username)
        elif mode == "Voice Chat": voice_chat_session(username)
        elif mode == "Video Chat": video_chat_session(username) # Call the new video session func

    start_chat_button.on_click(on_start_chat)
    with main_area:
        clear_output(wait=True); print("\nPlease select your preferred chat mode:")
        display(widgets.VBox([chat_mode, start_chat_button, chat_mode_output]))


# A. Text Chat Session (Keep as before, ensure it calls end_chat_procedure)
def text_chat_session(username):
    # (Keep the Text Chat session logic exactly as before)
    global chat_history
    text_input = widgets.Textarea(placeholder="Type...", description="You:")
    send_button = widgets.Button(description="Send")
    chat_log_output = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'scroll'})
    end_chat_button = widgets.Button(description="End & Report")

    with chat_log_output:
         clear_output(); initial_msg = "Bot: Text chat active. How can I help?"
         print(initial_msg); chat_history.append({'role': 'model', 'parts': [initial_msg]})
         if tts_engine: tts_q.put(initial_msg) # Optional TTS

    def on_send_clicked(b):
        user_msg = text_input.value; text_input.value = ""
        if not user_msg: return
        with chat_log_output: print(f"\nYou: {user_msg}")
        # Pass N/A for emotions in text mode
        bot_reply = get_chatbot_reply_final(user_msg, current_voice_emotion="N/A", current_facial_emotion="N/A")
        with chat_log_output: print(f"Bot: {bot_reply}")
        if tts_engine: tts_q.put(bot_reply)
        if "ending session" in bot_reply.lower(): end_chat_procedure(username)

    def on_end_chat_clicked(b): end_chat_procedure(username)

    send_button.on_click(on_send_clicked); end_chat_button.on_click(on_end_chat_clicked)
    text_chat_box = widgets.VBox([chat_log_output, text_input, send_button, end_chat_button])
    with main_area: clear_output(wait=True); print("--- Text Chat Session ---"); display(text_chat_box)


# B. Voice Chat Session (Keep as before, using SER/Spectrogram)
def voice_chat_session(username):
    # (Keep the Voice Chat session logic exactly as before)
    # It initializes and uses spectrogram_output_widget globally
    global chat_history, voice_emotion_log, audio_stream, stop_event, ser_fig, ser_ax, spectrogram_output_widget

    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc', 'margin_bottom': '10px'})
    chat_log_output_vc = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'overflow_y': 'scroll'})
    listen_button = widgets.Button(description="🎤 Speak", button_style='info', icon='microphone')
    end_chat_button_vc = widgets.Button(description="End & Report", button_style='danger')
    status_label = widgets.Label(value="Status: Initializing...")

    ser_fig, ser_ax = plt.subplots(figsize=(7, 4)); plt.close(ser_fig) # Setup plot fig
    ser_ax.set_title("Live Spectrogram - Waiting...")

    voice_chat_box = widgets.VBox([status_label, spectrogram_output_widget, chat_log_output_vc, widgets.HBox([listen_button, end_chat_button_vc])])
    with main_area:
        clear_output(wait=True); print("--- Voice Chat Session ---")
        with spectrogram_output_widget: display(ser_fig) # Show initial empty plot
        display(voice_chat_box)

    status_label.value = "Status: Starting threads..."
    threads = []
    stt_thread = threading.Thread(target=speech_to_text_thread); threads.append(stt_thread); stt_thread.start()
    ser_thread = threading.Thread(target=audio_processing_thread_ser); threads.append(ser_thread); ser_thread.start()
    if tts_engine: tts_thread = threading.Thread(target=text_to_speech_thread); threads.append(tts_thread); tts_thread.start()

    try:
        status_label.value = "Status: Starting audio..."
        audio_stream = sd.InputStream(samplerate=AUDIO_SAMPLE_RATE, channels=AUDIO_CHANNELS, blocksize=AUDIO_CHUNK_SAMPLES, callback=audio_callback, dtype='float32')
        audio_stream.start()
        status_label.value = "Status: Ready. Speak or press Speak button."
        with chat_log_output_vc: initial_msg="Bot: Voice chat active."; print(initial_msg)
        if tts_engine: tts_q.put(initial_msg)
        chat_history.append({'role': 'model', 'parts': [initial_msg]})
    except Exception as e: status_label.value = f"Status: ERROR starting audio: {e}"; print(f"Audio Error: {e}"); stop_event.set(); return

    def on_listen_button_clicked(b): listen_event.set(); status_label.value = "Status: Listening actively..."
    def on_end_vc_clicked(b): status_label.value = "Status: Ending..."; stop_event.set(); listen_button.disabled = True; end_chat_button_vc.disabled = True

    listen_button.on_click(on_listen_button_clicked)
    end_chat_button_vc.on_click(on_end_vc_clicked)

    # Main interaction loop is handled outside by run_prototype checking stt_text_q


# C. Integrated Video Chat Session (NEW IMPLEMENTATION!)
def video_chat_session(username):
    """Starts video, audio, SER, Spectrogram, STT, TTS, and interaction."""
    global chat_history, voice_emotion_log, face_emotion_log, audio_stream, video_capture, stop_event
    global ser_fig, ser_ax, spectrogram_output_widget, video_display_widget # Access global widget refs

    print("Initializing Video Chat Session...")

    # --- UI Setup ---
    status_label_vc = widgets.Label(value="Status: Initializing...")

    # Video display widget
    video_display_widget = widgets.Image(format='jpeg', width=480) # Adjust width as needed
    video_box = widgets.VBox([widgets.Label("Live Video Feed:"), video_display_widget], layout={'border': '1px solid grey'})

    # Spectrogram display widget
    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc'})
    spectrogram_box = widgets.VBox([widgets.Label("Live Audio Spectrogram:"), spectrogram_output_widget], layout={'margin_top': '10px'})
    # Initialize the plot figure and axes for spectrogram
    ser_fig, ser_ax = plt.subplots(figsize=(6, 3)) # Smaller spectrogram for video mode?
    ser_ax.set_title("Live Spectrogram - Waiting...")
    plt.close(ser_fig) # Close the initial display

    # Conversation log
    chat_log_output_vc = widgets.Output(layout={'border': '1px solid black', 'height': '250px', 'overflow_y': 'scroll', 'margin_top': '10px'})

    # Controls
    listen_button_vc = widgets.Button(description="🎤 Speak", button_style='info', icon='microphone')
    end_chat_button_vc = widgets.Button(description="End Video Chat & Report", button_style='danger')
    controls_box = widgets.HBox([listen_button_vc, end_chat_button_vc], layout={'margin_top': '10px'})

    # Arrange UI (e.g., Video and Spectrogram side-by-side, chat below)
    top_row = widgets.HBox([video_box, spectrogram_box])
    video_chat_main_box = widgets.VBox([status_label_vc, top_row, chat_log_output_vc, controls_box])

    # Display the UI
    with main_area:
        clear_output(wait=True)
        print("--- Video Chat Session ---")
        # Display placeholder plot initially in its widget
        with spectrogram_output_widget: display(ser_fig)
        display(video_chat_main_box)


    # --- Start Threads & Streams ---
    status_label_vc.value = "Status: Starting threads..."
    print("Starting threads for video session...")
    stop_event.clear()
    chat_history, voice_emotion_log, face_emotion_log = [], [], [] # Reset logs

    threads = []
    # Video Thread
    vid_thread = threading.Thread(target=video_capture_face_emotion_thread); threads.append(vid_thread); vid_thread.start()
    # Audio SER/Spectrogram Thread
    ser_thread = threading.Thread(target=audio_processing_thread_ser); threads.append(ser_thread); ser_thread.start()
    # STT Thread
    stt_thread = threading.Thread(target=speech_to_text_thread); threads.append(stt_thread); stt_thread.start()
    # TTS Thread
    if tts_engine: tts_thread = threading.Thread(target=text_to_speech_thread); threads.append(tts_thread); tts_thread.start()


    # Start Audio Stream
    try:
        status_label_vc.value = "Status: Starting audio..."
        audio_stream = sd.InputStream(samplerate=AUDIO_SAMPLE_RATE, channels=AUDIO_CHANNELS, blocksize=AUDIO_CHUNK_SAMPLES, callback=audio_callback, dtype='float32')
        audio_stream.start()
        status_label_vc.value = "Status: Ready. Speak or press Speak button."
        print("Audio stream started for video session.")

        with chat_log_output_vc: initial_msg="Bot: Video & audio chat active."; print(initial_msg)
        if tts_engine: tts_q.put(initial_msg)
        chat_history.append({'role': 'model', 'parts': [initial_msg]})

    except Exception as e:
        status_label_vc.value = f"Status: ERROR starting audio stream: {e}"
        print(f"Audio Stream Error: {e}"); stop_event.set(); return # Stop if audio fails


    # --- Button Actions ---
    def on_listen_vc_clicked(b): listen_event.set(); status_label_vc.value = "Status: Listening actively..."
    def on_end_vchat_clicked(b):
         status_label_vc.value = "Status: Ending...";
         stop_event.set(); # Signal threads & stream to stop
         listen_button_vc.disabled=True; end_chat_button_vc.disabled=True

    listen_button_vc.on_click(on_listen_vc_clicked)
    end_chat_button_vc.on_click(on_end_vchat_clicked)

    # Main interaction loop is handled outside by run_prototype checking stt_text_q


# --- Step 9: Session Report Generation (Updated to include Face Emotion) ---
def generate_report(username):
    # ... (keep report header, intake, initial assessment sections same as before) ...
    data = user_db[username]["data"]
    report_lines = []
    report_lines.append("="*40); report_lines.append("  Multi-Modal Chatbot Session Report"); report_lines.append("="*40)
    report_lines.append(f"User: {username}"); report_lines.append(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    report_lines.append("--- User Intake Summary ---")
    if data.get("intake"): [report_lines.append(f"  {k.capitalize()}: {v}") for k, v in data["intake"].items()]
    else: report_lines.append("  (No intake data)")
    report_lines.append("\n--- Initial Assessment (Experimental) ---")
    if data.get("issues"): [report_lines.append(f"  Potential for {k}: {int(v * 100)}% likelihood") for k, v in data["issues"].items()]
    else: report_lines.append("  (No assessment data)")

    # --- Add Voice Emotion Log ---
    report_lines.append("\n--- Voice Emotion Log ---")
    if voice_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence")
         report_lines.append("  ------------|-----------------|-----------")
         with voice_emotion_log_lock: # Access log safely
             log_limit = 30; display_log = voice_emotion_log[-log_limit:]
             if len(voice_emotion_log) > log_limit: report_lines.append("  ...(last 30 entries)...")
             for ts, emo, conf in display_log: report_lines.append(f"  {ts} | {emo:<15} | {conf:.2f}")
    else: report_lines.append("  (No voice emotions logged for this session)")

    # --- Add Face Emotion Log (NEW) ---
    report_lines.append("\n--- Face Emotion Log ---")
    if face_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence")
         report_lines.append("  ------------|-----------------|-----------")
         with face_emotion_log_lock: # Access log safely
             log_limit = 30; display_log = face_emotion_log[-log_limit:]
             if len(face_emotion_log) > log_limit: report_lines.append("  ...(last 30 entries)...")
             for ts, emo, conf in display_log: report_lines.append(f"  {ts} | {emo:<15} | {conf:.2f}")
    else: report_lines.append("  (No face emotions logged for this session)")
    # ------------------------------------

    report_lines.append("\n--- Conversation Transcript ---")
    if chat_history: [report_lines.append(f"  {t.get('role','?').capitalize()}: {t.get('parts',[''])[0]}") for t in chat_history]
    else: report_lines.append("  (No conversation recorded)")

    report_lines.append("\n"+"="*40)
    report_content = "\n".join(report_lines)
    filename = f"{username}_session_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    try:
        with open(filename, "w", encoding='utf-8') as f: f.write(report_content)
        print(f"\nSession report generated: '{filename}'")
    except Exception as e: print(f"\nError writing report: {e}")

    # --- Professional Help Suggestion (Same as before) ---
    issues = data.get("issues", {})
    if any(prob > 0.65 for prob in issues.values()):
         print("\nRecommendation: Consider reaching out to a mental health professional.")
         print("Resources: 988 Lifeline (US/Canada), Befrienders Worldwide, Crisis Text Line.")
    else: print("\nRecommendation: Continue self-care. Consult a professional if needed.")

# (Keep end_chat_procedure exactly as before - it stops threads/stream and calls generate_report)
def end_chat_procedure(username):
    print("\nEnding chat session and generating report...")
    stop_event.set()
    global audio_stream, video_capture # Ensure we clear global captures
    if audio_stream is not None and audio_stream.active:
        try: print("Stopping audio stream..."); audio_stream.stop(); audio_stream.close(); print("Audio stopped.")
        except Exception as e: print(f"Error stopping audio: {e}")
        finally: audio_stream = None
    if video_capture is not None and video_capture.isOpened(): # Add video cleanup
        try: print("Releasing video capture..."); video_capture.release(); print("Video released.")
        except Exception as e: print(f"Error releasing video: {e}")
        finally: video_capture = None
    cv2.destroyAllWindows() # Close any OpenCV windows just in case
    for i in range(3): cv2.waitKey(1)
    print("Waiting briefly for threads...")
    time.sleep(1.5) # Slightly longer wait?
    with main_area:
        clear_output(wait=True); generate_report(username)
        print("\nSession ended. Run the first cell again to log in.")


# --- Step 10: Main Application Execution Logic (Updated Interaction Loop) ---
def run_prototype():
    clear_output(); print("="*50); print(" Multi-Modal Mental Health Chatbot Prototype"); print("="*50)
    display(login_box)
    display(main_area)
    print("\nPrototype running. Login to start...")
    try:
         print("\nPrototype running. Login to start.")
         while not stop_event.is_set():
            try:
                 user_text = stt_text_q.get(timeout=0.5)

                 # Find appropriate chat log widget (Needs better state mgmt)
                 current_chat_log_widget = main_area # Default to main

                 # --- Process User Speech ---
                 if user_text:
                     print(f"\n[STT Received]: {user_text}") # Debug
                     with current_chat_log_widget: # Display in the main area for now
                          print(f"You (voice): {user_text}")

                     # Fetch latest emotions (use globals directly - read is often okay without lock if writes are atomic/infrequent)
                     # Use defaults if not updated yet
                     current_voice_emotion = last_detected_voice_emotion_global
                     current_facial_emotion = last_detected_face_emotion_global

                     # Get bot reply using both emotions
                     bot_reply = get_chatbot_reply_final(user_text, current_voice_emotion, current_facial_emotion)

                     with current_chat_log_widget: # Display bot reply
                         print(f"Bot: {bot_reply}")
                     if tts_engine: tts_q.put(bot_reply)

                     # Check if the bot response signals an end
                     if "ending session" in bot_reply.lower():
                           print("Bot initiated session end.")
                           # Ensure TTS finishes before full stop if possible
                           if tts_engine: tts_q.join()
                           stop_event.set() # Signal threads after response processed

                 stt_text_q.task_done() # Mark STT task as done

            except queue.Empty: time.sleep(0.1); continue # No text, just wait
            except Exception as main_loop_e: print(f"!! Error in main loop: {main_loop_e} !!"); time.sleep(1)

    except KeyboardInterrupt: print("\nKeyboard Interrupt! Stopping..."); stop_event.set()
    finally:
         print("Main loop exited.")
         if not stop_event.is_set(): # Ensure cleanup if loop broke unexpectedly
            stop_event.set()
         # Run cleanup directly if needed (end_chat_procedure might be better called explicitly via button)
         # Cleanup shared resources that might linger
         global audio_stream, video_capture
         if audio_stream is not None: audio_stream.close(); audio_stream = None
         if video_capture is not None: video_capture.release(); video_capture = None
         cv2.destroyAllWindows()
         print("Prototype Shutdown Complete.")


# --- Start the Prototype ---
run_prototype()

 Multi-Modal Mental Health Chatbot Prototype


Output()


Prototype running. Login to start.
